<a href="https://colab.research.google.com/github/FrancoR72/GC-MS_compounds_first_alignment/blob/main/Copia_di_Untitled8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# COSTRUZIONE AUTOMATICA DELLA CACHE SENSORIALE
# Versione corretta per una singola cella Google Colab
#
# INPUT:
#   file .xlsx contenente il foglio Compound_Mapping oppure un unico
#   foglio con almeno:
#       IUPAC_name
#       CAS
#
#   Colonna facoltativa:
#       Compound_role
#
# OUTPUT:
#   Sensory_Cache.xlsx
#
# Il codice:
#   1. legge la chiave API dai Secrets di Google Colab;
#   2. carica l'elenco dei composti;
#   3. verifica formalmente i numeri CAS;
#   4. controlla l'identità tramite PubChem;
#   5. ricerca online soglie olfattive in aria e descrittori;
#   6. converte le soglie in µg/m³;
#   7. crea un file Excel con dati, fonti e record da revisionare.
# =====================================================================


# =====================================================================
# 0. INSTALLAZIONE DELLE LIBRERIE
# =====================================================================

!pip -q install --upgrade openai pydantic xlsxwriter


# =====================================================================
# 1. IMPORTAZIONI
# =====================================================================

import re
import json
import time
import getpass
from datetime import date
from itertools import zip_longest
from typing import Optional, List

import numpy as np
import pandas as pd
import requests

from pydantic import BaseModel, Field
from openai import OpenAI
from google.colab import files, userdata


# =====================================================================
# 2. PARAMETRI MODIFICABILI
# =====================================================================

# Modello OpenAI.
# gpt-5.6 è adatto alla ricerca web complessa.
MODEL = "gpt-5.6"

# Nome preferenziale del foglio di input.
INPUT_SHEET_PREFERRED = "Compound_Mapping"

# Nome del file prodotto.
OUTPUT_FILE = "Sensory_Cache.xlsx"

# Intestazioni richieste nel file Excel.
IUPAC_COLUMN = "IUPAC_name"
CAS_COLUMN = "CAS"

# Intestazione facoltativa per distinguere analiti e standard interno.
ROLE_COLUMN = "Compound_role"

# Pausa fra una ricerca e la successiva.
PAUSE_SECONDS = 1.0

# Numero massimo di tentativi per ogni composto.
MAX_RETRIES = 2

# Volume molare approssimato a 25 °C e 1 atm.
MOLAR_VOLUME_L_MOL = 24.45

# Famiglie sensoriali ammesse.
ALLOWED_FAMILIES = [
    "Cacao/Cioccolato",
    "Tostato",
    "Caffè",
    "Frutta secca",
    "Caramellato",
    "Dolce",
    "Fruttato",
    "Floreale",
    "Verde/Erbaceo",
    "Speziato",
    "Legnoso",
    "Affumicato",
    "Terroso",
    "Fungino",
    "Lattico",
    "Grasso/Ceroso",
    "Fermentato",
    "Solforato",
    "Animale",
    "Chimico/Solvente",
    "Fenolico/Medicinale",
    "Altro",
    "Non classificabile"
]


# =====================================================================
# 3. LETTURA DELLA CHIAVE API
# =====================================================================

# Prima prova a leggere il Secret di Colab.
try:
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = None

# Se il Secret non è disponibile, consente comunque l'inserimento manuale.
if not api_key:
    print(
        "Il Secret OPENAI_API_KEY non è stato trovato o non è accessibile."
    )
    api_key = getpass.getpass(
        "Inserire la chiave API OpenAI. "
        "La chiave non sarà visualizzata: "
    )

if not api_key or not api_key.strip():
    raise ValueError(
        "La chiave API OpenAI non è disponibile."
    )

client = OpenAI(api_key=api_key.strip())

print("Chiave API caricata correttamente.")


# =====================================================================
# 4. STRUTTURA DELLA RISPOSTA DEL MODELLO
# =====================================================================

class SensoryRecord(BaseModel):

    cas_input: str
    iupac_input: str

    identity_match: str = Field(
        description=(
            "Valori consentiti: confirmed, probable, "
            "conflicting, not_found"
        )
    )

    common_name: Optional[str] = None
    molecular_formula: Optional[str] = None
    molecular_weight_g_mol: Optional[float] = None

    threshold_air_found: bool = False

    threshold_air_value: Optional[float] = None
    threshold_air_unit_original: Optional[str] = None

    threshold_air_min_original: Optional[float] = None
    threshold_air_max_original: Optional[float] = None

    threshold_type: Optional[str] = Field(
        default=None,
        description=(
            "Detection, recognition, unspecified oppure null"
        )
    )

    threshold_medium: Optional[str] = Field(
        default=None,
        description="air oppure null"
    )

    threshold_conditions: Optional[str] = None

    odor_descriptors: List[str] = Field(
        default_factory=list
    )

    flavor_descriptors: List[str] = Field(
        default_factory=list
    )

    sensory_families: List[str] = Field(
        default_factory=list
    )

    threshold_source_name: Optional[str] = None
    threshold_source_url: Optional[str] = None
    threshold_reference: Optional[str] = None

    descriptor_source_names: List[str] = Field(
        default_factory=list
    )

    descriptor_source_urls: List[str] = Field(
        default_factory=list
    )

    confidence: str = Field(
        description="Valori consentiti: high, medium, low"
    )

    review_required: bool = False
    notes: Optional[str] = None


# =====================================================================
# 5. FUNZIONI DI PULIZIA E CONTROLLO
# =====================================================================

def clean_text(value):
    """
    Elimina spazi iniziali e finali mantenendo i valori mancanti.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text == "" or text.lower() == "nan":
        return np.nan

    return text


def normalize_cas(value):
    """
    Normalizza la scrittura del numero CAS.
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    value = (
        value
        .replace("–", "-")
        .replace("—", "-")
        .replace("−", "-")
    )

    value = re.sub(r"\s+", "", value)

    return value


def validate_cas(cas_number):
    """
    Controlla:
    - formato del CAS;
    - cifra finale di controllo.
    """

    if pd.isna(cas_number):
        return False

    cas_number = str(cas_number).strip()

    if not re.fullmatch(
        r"\d{2,7}-\d{2}-\d",
        cas_number
    ):
        return False

    digits = cas_number.replace("-", "")

    body = digits[:-1]
    expected_check_digit = int(digits[-1])

    calculated_sum = sum(
        position * int(digit)
        for position, digit
        in enumerate(reversed(body), start=1)
    )

    calculated_check_digit = calculated_sum % 10

    return calculated_check_digit == expected_check_digit


def normalize_role(value):
    """
    Normalizza il ruolo dell'analita.
    """

    if pd.isna(value):
        return ""

    return (
        str(value)
        .strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
    )


def select_input_sheet(filename):
    """
    Seleziona:
    1. Compound_Mapping, se presente;
    2. l'unico foglio disponibile;
    3. genera errore se esistono più fogli e nessuno ha il nome atteso.
    """

    excel_file = pd.ExcelFile(filename)

    if INPUT_SHEET_PREFERRED in excel_file.sheet_names:
        return INPUT_SHEET_PREFERRED

    if len(excel_file.sheet_names) == 1:
        selected = excel_file.sheet_names[0]

        print(
            f"Il foglio '{INPUT_SHEET_PREFERRED}' non è presente. "
            f"Verrà utilizzato l'unico foglio disponibile: "
            f"'{selected}'."
        )

        return selected

    raise ValueError(
        f"Il foglio '{INPUT_SHEET_PREFERRED}' non è presente.\n"
        f"Fogli disponibili: {excel_file.sheet_names}"
    )


def safe_join(values, separator="; "):
    """
    Unisce una lista di stringhe eliminando valori vuoti e duplicati.
    """

    if not values:
        return ""

    cleaned = []

    for value in values:

        if value is None:
            continue

        text = str(value).strip()

        if text and text not in cleaned:
            cleaned.append(text)

    return separator.join(cleaned)


# =====================================================================
# 6. CONTROLLO DELL'IDENTITÀ MEDIANTE PUBCHEM
# =====================================================================

def get_pubchem_identity(cas_number):
    """
    Recupera da PubChem:
    - CID;
    - nome IUPAC;
    - formula molecolare;
    - massa molecolare.

    Se il CAS non viene trovato restituisce un dizionario vuoto.
    """

    encoded_cas = requests.utils.quote(
        str(cas_number),
        safe=""
    )

    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
        f"compound/name/{encoded_cas}/property/"
        "IUPACName,MolecularFormula,MolecularWeight/JSON"
    )

    try:

        response = requests.get(
            url,
            timeout=30
        )

        if response.status_code == 404:
            return {}

        response.raise_for_status()

        payload = response.json()

        properties = (
            payload
            .get("PropertyTable", {})
            .get("Properties", [])
        )

        if not properties:
            return {}

        record = properties[0]

        cid = record.get("CID")

        return {
            "PubChem_CID": cid,
            "PubChem_IUPAC_name": record.get("IUPACName"),
            "PubChem_formula": record.get("MolecularFormula"),
            "PubChem_MW": record.get("MolecularWeight"),
            "PubChem_URL": (
                f"https://pubchem.ncbi.nlm.nih.gov/compound/{cid}"
                if cid is not None
                else None
            ),
            "PubChem_error": None
        }

    except Exception as error:

        return {
            "PubChem_CID": None,
            "PubChem_IUPAC_name": None,
            "PubChem_formula": None,
            "PubChem_MW": None,
            "PubChem_URL": None,
            "PubChem_error": str(error)
        }


# =====================================================================
# 7. NORMALIZZAZIONE DELLE UNITÀ
# =====================================================================

def normalize_unit_string(unit):
    """
    Normalizza varianti come:

    ng/L_air
    ng/L (air)
    ng/L air
    ng/lair
    µg/m³
    """

    if unit is None or pd.isna(unit):
        return None

    unit_clean = str(unit).lower().strip()

    replacements = {
        "μ": "µ",
        "³": "3",
        "_": "",
        "(": "",
        ")": "",
        "[": "",
        "]": "",
        "{": "",
        "}": "",
        " ": "",
        "litres": "l",
        "litre": "l",
        "liters": "l",
        "liter": "l",
        "cubicmetre": "m3",
        "cubicmeter": "m3",
        "m^3": "m3"
    }

    for old, new in replacements.items():
        unit_clean = unit_clean.replace(old, new)

    return unit_clean


def threshold_to_ug_m3(
    value,
    unit,
    molecular_weight=None
):
    """
    Converte una soglia olfattiva in aria in µg/m³.

    Conversioni supportate:
    - µg/m³
    - mg/m³
    - ng/m³
    - pg/m³
    - ng/L aria
    - µg/L aria
    - pg/L aria
    - ppb / ppbv
    - ppm / ppmv
    - ppt / pptv

    Per ppb, ppm e ppt serve la massa molecolare.
    """

    if value is None or unit is None:
        return np.nan

    try:
        value = float(value)
    except (TypeError, ValueError):
        return np.nan

    if not np.isfinite(value):
        return np.nan

    unit_clean = normalize_unit_string(unit)

    if unit_clean is None:
        return np.nan

    # µg/m³
    if unit_clean in {
        "µg/m3",
        "ug/m3",
        "microgram/m3",
        "micrograms/m3"
    }:
        return value

    # mg/m³
    if unit_clean in {
        "mg/m3",
        "milligram/m3",
        "milligrams/m3"
    }:
        return value * 1000.0

    # ng/m³
    if unit_clean in {
        "ng/m3",
        "nanogram/m3",
        "nanograms/m3"
    }:
        return value / 1000.0

    # pg/m³
    if unit_clean in {
        "pg/m3",
        "picogram/m3",
        "picograms/m3"
    }:
        return value / 1_000_000.0

    # 1 ng/L = 1 µg/m³
    if unit_clean in {
        "ng/l",
        "ng/lair",
        "ng/l-air",
        "nanogram/l",
        "nanograms/l"
    }:
        return value

    # 1 µg/L = 1000 µg/m³
    if unit_clean in {
        "µg/l",
        "ug/l",
        "µg/lair",
        "ug/lair",
        "microgram/l",
        "micrograms/l"
    }:
        return value * 1000.0

    # 1 pg/L = 0,001 µg/m³
    if unit_clean in {
        "pg/l",
        "pg/lair",
        "picogram/l",
        "picograms/l"
    }:
        return value / 1000.0

    # Per le unità volumetriche è necessaria la massa molecolare.
    try:
        mw = float(molecular_weight)
    except (TypeError, ValueError):
        mw = np.nan

    if pd.isna(mw) or mw <= 0:
        return np.nan

    # ppbv → µg/m³
    if unit_clean in {
        "ppb",
        "ppbv",
        "partperbillion",
        "partsperbillion"
    }:
        return value * mw / MOLAR_VOLUME_L_MOL

    # ppmv → µg/m³
    if unit_clean in {
        "ppm",
        "ppmv",
        "partpermillion",
        "partspermillion"
    }:
        return value * mw * 1000.0 / MOLAR_VOLUME_L_MOL

    # pptv → µg/m³
    if unit_clean in {
        "ppt",
        "pptv",
        "partpertrillion",
        "partspertrillion"
    }:
        return value * mw / (
            MOLAR_VOLUME_L_MOL * 1000.0
        )

    return np.nan


# =====================================================================
# 8. FUNZIONE DI RICERCA ONLINE
# =====================================================================

def search_compound_online(
    cas_number,
    iupac_name,
    pubchem_data
):
    """
    Ricerca:
    - soglia olfattiva esplicitamente riferita all'aria;
    - descrittori olfattivi;
    - descrittori flavour;
    - famiglie sensoriali;
    - fonti.

    Restituisce un oggetto SensoryRecord.
    """

    pubchem_context = json.dumps(
        pubchem_data,
        ensure_ascii=False,
        indent=2
    )

    allowed_families_text = ", ".join(
        ALLOWED_FAMILIES
    )

    system_prompt = f"""
Sei un ricercatore esperto in chimica degli aromi,
HS-SPME-GC-MS, gascromatografia-olfattometria e soglie olfattive.

Devi compilare una scheda strutturata e documentabile per un composto.

FONTI DA PRIVILEGIARE:
1. pubblicazioni scientifiche originali;
2. compilazioni di van Gemert o Leffingwell;
3. The Good Scents Company;
4. PubChem;
5. FlavorDB e altre banche dati scientifiche pertinenti.

REGOLE OBBLIGATORIE:

- Verifica che il CAS e il nome fornito corrispondano alla stessa sostanza.
- Distingui chiaramente soglia in aria, acqua, olio, solvente
  e matrice alimentare.
- Accetta nel campo threshold_air esclusivamente un valore
  dichiarato esplicitamente come soglia olfattiva in aria.
- Non trasformare una soglia in acqua in una soglia in aria.
- Non usare come soglia la concentrazione impiegata per una
  valutazione descrittiva in solvente.
- Distingui detection threshold e recognition threshold.
- Conserva l'unità originale come riportata dalla fonte.
- Se esistono più valori in aria, scegli un valore rappresentativo
  soltanto se la scelta è difendibile.
- Quando possibile, registra anche minimo e massimo.
- Se la soglia in aria non è reperibile, imposta:
      threshold_air_found = false
      threshold_air_value = null
      threshold_air_unit_original = null
- Non inventare valori, fonti o URL.
- Gli URL devono essere reali e riferirsi alle pagine consultate.
- I descrittori devono essere brevi termini inglesi.
- Separa odor descriptors da flavor descriptors.
- Le famiglie sensoriali devono essere scelte esclusivamente da:
  {allowed_families_text}
- Imposta review_required = true quando:
    * le fonti sono discordanti;
    * l'identificazione è incerta;
    * il dato riguarda uno stereoisomero non specificato;
    * non è chiaro se la soglia sia in aria;
    * la fonte non è sufficientemente documentata.
"""

    user_prompt = f"""
COMPOSTO DA STUDIARE

Nome IUPAC fornito:
{iupac_name}

CAS fornito:
{cas_number}

Informazioni preliminari ottenute da PubChem:
{pubchem_context}

Ricercare sul web:

1. soglia olfattiva esplicitamente misurata in aria;
2. tipo di soglia: detection, recognition o unspecified;
3. condizioni sperimentali, se disponibili;
4. descrittori olfattivi;
5. descrittori aromatici o flavour;
6. famiglie sensoriali;
7. fonti e riferimenti.

Non inserire soglie in acqua nel campo threshold_air.
"""

    response = client.responses.parse(
        model=MODEL,
        tools=[
            {
                "type": "web_search",
                "filters": {
                    "allowed_domains": [
                        "pubmed.ncbi.nlm.nih.gov",
                        "pmc.ncbi.nlm.nih.gov",
                        "pubchem.ncbi.nlm.nih.gov",
                        "thegoodscentscompany.com",
                        "leffingwell.com",
                        "sciencedirect.com",
                        "acs.org",
                        "springer.com",
                        "wiley.com",
                        "tandfonline.com"
                    ]
                }
            }
        ],
        input=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        text_format=SensoryRecord
    )

    if response.output_parsed is None:
        raise ValueError(
            "La risposta API non contiene un record strutturato."
        )

    return response.output_parsed


# =====================================================================
# 9. CARICAMENTO DEL FILE DI INPUT
# =====================================================================

print()
print("=" * 72)
print("CARICAMENTO DEL FILE")
print("=" * 72)
print(
    "Caricare il file Excel contenente IUPAC_name e CAS."
)

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError(
        "È necessario caricare un solo file Excel."
    )

input_filename = next(iter(uploaded))

sheet_name = select_input_sheet(
    input_filename
)

print()
print("File caricato:", input_filename)
print("Foglio utilizzato:", sheet_name)

mapping_df = pd.read_excel(
    input_filename,
    sheet_name=sheet_name,
    dtype={CAS_COLUMN: str}
)


# =====================================================================
# 10. CONTROLLO DELLA STRUTTURA DEL FILE
# =====================================================================

required_columns = [
    IUPAC_COLUMN,
    CAS_COLUMN
]

missing_columns = [
    column
    for column in required_columns
    if column not in mapping_df.columns
]

if missing_columns:
    raise ValueError(
        "Mancano le colonne obbligatorie: "
        + ", ".join(missing_columns)
    )

mapping_df[IUPAC_COLUMN] = (
    mapping_df[IUPAC_COLUMN]
    .apply(clean_text)
)

mapping_df[CAS_COLUMN] = (
    mapping_df[CAS_COLUMN]
    .apply(normalize_cas)
)

mapping_df["CAS_valid"] = (
    mapping_df[CAS_COLUMN]
    .apply(validate_cas)
)


# =====================================================================
# 11. ESCLUSIONE DELLO STANDARD INTERNO
# =====================================================================

if ROLE_COLUMN in mapping_df.columns:

    normalized_roles = (
        mapping_df[ROLE_COLUMN]
        .apply(normalize_role)
    )

    internal_standard_labels = {
        "internal standard",
        "internalstandard",
        "standard interno",
        "internal std",
        "is"
    }

    analytes_df = mapping_df.loc[
        ~normalized_roles.isin(
            internal_standard_labels
        )
    ].copy()

else:
    analytes_df = mapping_df.copy()


# Elimina righe prive di identificativi.
analytes_df = analytes_df.loc[
    analytes_df[IUPAC_COLUMN].notna()
    & analytes_df[CAS_COLUMN].notna()
].copy()

# Elimina CAS duplicati.
analytes_df = analytes_df.drop_duplicates(
    subset=[CAS_COLUMN],
    keep="first"
).reset_index(drop=True)

if analytes_df.empty:
    raise ValueError(
        "Non sono presenti composti validi da elaborare."
    )

print()
print(
    f"Composti da elaborare: {len(analytes_df)}"
)


# =====================================================================
# 12. TABELLE VUOTE PER I RISULTATI
# =====================================================================

records = []
threshold_source_rows = []
descriptor_source_rows = []


# =====================================================================
# 13. ELABORAZIONE DEI COMPOSTI
# =====================================================================

for index, row in analytes_df.iterrows():

    cas_number = row[CAS_COLUMN]
    iupac_name = row[IUPAC_COLUMN]
    cas_valid = bool(row["CAS_valid"])

    print()
    print("=" * 72)
    print(
        f"[{index + 1}/{len(analytes_df)}] "
        f"{iupac_name} — {cas_number}"
    )

    # ---------------------------------------------------------------
    # CAS non valido
    # ---------------------------------------------------------------

    if not cas_valid:

        print(
            "CAS formalmente non valido: "
            "la ricerca online non verrà eseguita."
        )

        records.append(
            {
                "IUPAC_name_input": iupac_name,
                "CAS": cas_number,
                "CAS_valid": False,
                "Identity_match": "conflicting",
                "Common_name": "",
                "Molecular_formula": "",
                "Molecular_weight_g_mol": np.nan,
                "Threshold_air_found": False,
                "Odor_threshold_air_value_original": np.nan,
                "Odor_threshold_air_unit_original": "",
                "Odor_threshold_air_min_original": np.nan,
                "Odor_threshold_air_max_original": np.nan,
                "Odor_threshold_air_ug_m3": np.nan,
                "Threshold_conversion_status": "Non eseguita",
                "Threshold_type": "",
                "Threshold_conditions": "",
                "Odor_descriptors": "",
                "Flavor_descriptors": "",
                "Sensory_family": "",
                "Threshold_source_name": "",
                "Threshold_source_url": "",
                "Threshold_reference": "",
                "Descriptor_source_names": "",
                "Descriptor_source_urls": "",
                "Confidence": "low",
                "Review_required": True,
                "Notes": "Numero CAS formalmente non valido",
                "PubChem_CID": np.nan,
                "PubChem_URL": "",
                "PubChem_error": "",
                "Retrieved_on": str(date.today())
            }
        )

        continue

    # ---------------------------------------------------------------
    # Controllo identità su PubChem
    # ---------------------------------------------------------------

    pubchem_data = get_pubchem_identity(
        cas_number
    )

    sensory = None
    last_error = None

    # ---------------------------------------------------------------
    # Ricerca API con tentativi automatici
    # ---------------------------------------------------------------

    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):

        try:

            if attempt > 1:
                print(
                    f"Nuovo tentativo API "
                    f"({attempt}/{MAX_RETRIES})..."
                )

            sensory = search_compound_online(
                cas_number=cas_number,
                iupac_name=iupac_name,
                pubchem_data=pubchem_data
            )

            break

        except Exception as error:

            last_error = error

            print(
                f"Errore API al tentativo "
                f"{attempt}/{MAX_RETRIES}: {error}"
            )

            if attempt < MAX_RETRIES:
                time.sleep(3)

    # ---------------------------------------------------------------
    # Errore definitivo
    # ---------------------------------------------------------------

    if sensory is None:

        records.append(
            {
                "IUPAC_name_input": iupac_name,
                "CAS": cas_number,
                "CAS_valid": True,
                "Identity_match": "not_found",
                "Common_name": "",
                "Molecular_formula": (
                    pubchem_data.get("PubChem_formula") or ""
                ),
                "Molecular_weight_g_mol": (
                    pubchem_data.get("PubChem_MW")
                ),
                "Threshold_air_found": False,
                "Odor_threshold_air_value_original": np.nan,
                "Odor_threshold_air_unit_original": "",
                "Odor_threshold_air_min_original": np.nan,
                "Odor_threshold_air_max_original": np.nan,
                "Odor_threshold_air_ug_m3": np.nan,
                "Threshold_conversion_status": "Non eseguita",
                "Threshold_type": "",
                "Threshold_conditions": "",
                "Odor_descriptors": "",
                "Flavor_descriptors": "",
                "Sensory_family": "",
                "Threshold_source_name": "",
                "Threshold_source_url": "",
                "Threshold_reference": "",
                "Descriptor_source_names": "",
                "Descriptor_source_urls": "",
                "Confidence": "low",
                "Review_required": True,
                "Notes": (
                    "Errore durante la ricerca online: "
                    f"{last_error}"
                ),
                "PubChem_CID": pubchem_data.get(
                    "PubChem_CID"
                ),
                "PubChem_URL": (
                    pubchem_data.get("PubChem_URL") or ""
                ),
                "PubChem_error": (
                    pubchem_data.get("PubChem_error") or ""
                ),
                "Retrieved_on": str(date.today())
            }
        )

        time.sleep(PAUSE_SECONDS)
        continue

    # ---------------------------------------------------------------
    # Massa molecolare
    # ---------------------------------------------------------------

    molecular_weight = (
        sensory.molecular_weight_g_mol
    )

    if molecular_weight is None:
        molecular_weight = pubchem_data.get(
            "PubChem_MW"
        )

    try:
        molecular_weight = float(
            molecular_weight
        )
    except (TypeError, ValueError):
        molecular_weight = np.nan

    # ---------------------------------------------------------------
    # Conversione soglia
    # ---------------------------------------------------------------

    threshold_ug_m3 = threshold_to_ug_m3(
        value=sensory.threshold_air_value,
        unit=sensory.threshold_air_unit_original,
        molecular_weight=molecular_weight
    )

    if not sensory.threshold_air_found:

        conversion_status = (
            "Soglia in aria non disponibile"
        )

    elif pd.notna(threshold_ug_m3):

        conversion_status = (
            "Conversione eseguita"
        )

    else:

        conversion_status = (
            "Unità non riconosciuta o massa molecolare mancante"
        )

        sensory.review_required = True

    # ---------------------------------------------------------------
    # Preparazione del record principale
    # ---------------------------------------------------------------

    odor_descriptors = safe_join(
        sensory.odor_descriptors
    )

    flavor_descriptors = safe_join(
        sensory.flavor_descriptors
    )

    sensory_families = safe_join(
        sensory.sensory_families,
        separator=" | "
    )

    descriptor_names = safe_join(
        sensory.descriptor_source_names
    )

    descriptor_urls = safe_join(
        sensory.descriptor_source_urls
    )

    records.append(
        {
            "IUPAC_name_input": iupac_name,
            "CAS": cas_number,
            "CAS_valid": True,

            "Identity_match": sensory.identity_match,
            "Common_name": sensory.common_name or "",

            "Molecular_formula": (
                sensory.molecular_formula
                or pubchem_data.get("PubChem_formula")
                or ""
            ),

            "Molecular_weight_g_mol": molecular_weight,

            "Threshold_air_found": (
                sensory.threshold_air_found
            ),

            "Odor_threshold_air_value_original": (
                sensory.threshold_air_value
            ),

            "Odor_threshold_air_unit_original": (
                sensory.threshold_air_unit_original or ""
            ),

            "Odor_threshold_air_min_original": (
                sensory.threshold_air_min_original
            ),

            "Odor_threshold_air_max_original": (
                sensory.threshold_air_max_original
            ),

            "Odor_threshold_air_ug_m3": threshold_ug_m3,

            "Threshold_conversion_status": (
                conversion_status
            ),

            "Threshold_type": (
                sensory.threshold_type or ""
            ),

            "Threshold_conditions": (
                sensory.threshold_conditions or ""
            ),

            "Odor_descriptors": odor_descriptors,
            "Flavor_descriptors": flavor_descriptors,
            "Sensory_family": sensory_families,

            "Threshold_source_name": (
                sensory.threshold_source_name or ""
            ),

            "Threshold_source_url": (
                sensory.threshold_source_url or ""
            ),

            "Threshold_reference": (
                sensory.threshold_reference or ""
            ),

            "Descriptor_source_names": (
                descriptor_names
            ),

            "Descriptor_source_urls": (
                descriptor_urls
            ),

            "Confidence": sensory.confidence,

            "Review_required": (
                sensory.review_required
            ),

            "Notes": sensory.notes or "",

            "PubChem_CID": pubchem_data.get(
                "PubChem_CID"
            ),

            "PubChem_URL": (
                pubchem_data.get("PubChem_URL") or ""
            ),

            "PubChem_error": (
                pubchem_data.get("PubChem_error") or ""
            ),

            "Retrieved_on": str(date.today())
        }
    )

    # ---------------------------------------------------------------
    # Tabella separata delle fonti delle soglie
    # ---------------------------------------------------------------

    if (
        sensory.threshold_source_name
        or sensory.threshold_source_url
    ):

        threshold_source_rows.append(
            {
                "CAS": cas_number,
                "IUPAC_name_input": iupac_name,
                "Threshold_value": (
                    sensory.threshold_air_value
                ),
                "Threshold_unit": (
                    sensory.threshold_air_unit_original
                ),
                "Threshold_value_ug_m3": threshold_ug_m3,
                "Threshold_min_original": (
                    sensory.threshold_air_min_original
                ),
                "Threshold_max_original": (
                    sensory.threshold_air_max_original
                ),
                "Threshold_type": sensory.threshold_type,
                "Threshold_medium": (
                    sensory.threshold_medium
                ),
                "Source_name": (
                    sensory.threshold_source_name
                ),
                "Source_URL": (
                    sensory.threshold_source_url
                ),
                "Reference": (
                    sensory.threshold_reference
                ),
                "Conditions": (
                    sensory.threshold_conditions
                ),
                "Retrieved_on": str(date.today())
            }
        )

    # ---------------------------------------------------------------
    # Tabella separata delle fonti dei descrittori
    # ---------------------------------------------------------------

    for source_name, source_url in zip_longest(
        sensory.descriptor_source_names,
        sensory.descriptor_source_urls,
        fillvalue=""
    ):

        if source_name or source_url:

            descriptor_source_rows.append(
                {
                    "CAS": cas_number,
                    "IUPAC_name_input": iupac_name,
                    "Source_name": source_name,
                    "Source_URL": source_url,
                    "Retrieved_on": str(date.today())
                }
            )

    # ---------------------------------------------------------------
    # Informazioni visualizzate durante l'esecuzione
    # ---------------------------------------------------------------

    print(
        "Soglia in aria:",
        sensory.threshold_air_value,
        sensory.threshold_air_unit_original
    )

    print(
        "Soglia convertita:",
        threshold_ug_m3,
        "µg/m³"
    )

    print(
        "Descrittori olfattivi:",
        odor_descriptors
    )

    if conversion_status != "Conversione eseguita":
        print(
            "ATTENZIONE:",
            conversion_status
        )

    time.sleep(PAUSE_SECONDS)


# =====================================================================
# 14. CREAZIONE DELLE TABELLE
# =====================================================================

cache_df = pd.DataFrame(records)

threshold_source_columns = [
    "CAS",
    "IUPAC_name_input",
    "Threshold_value",
    "Threshold_unit",
    "Threshold_value_ug_m3",
    "Threshold_min_original",
    "Threshold_max_original",
    "Threshold_type",
    "Threshold_medium",
    "Source_name",
    "Source_URL",
    "Reference",
    "Conditions",
    "Retrieved_on"
]

descriptor_source_columns = [
    "CAS",
    "IUPAC_name_input",
    "Source_name",
    "Source_URL",
    "Retrieved_on"
]

threshold_sources_df = pd.DataFrame(
    threshold_source_rows,
    columns=threshold_source_columns
)

descriptor_sources_df = pd.DataFrame(
    descriptor_source_rows,
    columns=descriptor_source_columns
)


# Record che richiedono controllo manuale.
review_mask = (
    cache_df["Review_required"].fillna(True)
    |
    ~cache_df["Threshold_air_found"].fillna(False)
    |
    cache_df["Odor_threshold_air_ug_m3"].isna()
    |
    (cache_df["Identity_match"] != "confirmed")
)

review_df = cache_df.loc[
    review_mask
].copy()


# Riepilogo.
summary_df = pd.DataFrame(
    {
        "Indicatore": [
            "Composti caricati",
            "CAS formalmente validi",
            "Soglie in aria trovate",
            "Soglie convertite in µg/m³",
            "Record da revisionare",
            "Modello OpenAI utilizzato",
            "Data di elaborazione"
        ],
        "Valore": [
            len(cache_df),
            int(
                cache_df["CAS_valid"]
                .fillna(False)
                .sum()
            ),
            int(
                cache_df["Threshold_air_found"]
                .fillna(False)
                .sum()
            ),
            int(
                cache_df["Odor_threshold_air_ug_m3"]
                .notna()
                .sum()
            ),
            len(review_df),
            MODEL,
            str(date.today())
        ]
    }
)


# =====================================================================
# 15. SALVATAGGIO CON XLSXWRITER
# =====================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="xlsxwriter"
) as writer:

    cache_df.to_excel(
        writer,
        sheet_name="Sensory_Cache",
        index=False
    )

    threshold_sources_df.to_excel(
        writer,
        sheet_name="Threshold_Sources",
        index=False
    )

    descriptor_sources_df.to_excel(
        writer,
        sheet_name="Descriptor_Sources",
        index=False
    )

    review_df.to_excel(
        writer,
        sheet_name="Manual_Review",
        index=False
    )

    analytes_df.to_excel(
        writer,
        sheet_name="Input_Compounds",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Processing_Summary",
        index=False
    )

    # ---------------------------------------------------------------
    # Formattazione del workbook
    # ---------------------------------------------------------------

    workbook = writer.book

    header_format = workbook.add_format(
        {
            "bold": True,
            "bg_color": "#D9EAF7",
            "border": 1,
            "text_wrap": True,
            "valign": "top"
        }
    )

    wrap_format = workbook.add_format(
        {
            "text_wrap": True,
            "valign": "top"
        }
    )

    numeric_format = workbook.add_format(
        {
            "num_format": "0.000000",
            "valign": "top"
        }
    )

    scientific_format = workbook.add_format(
        {
            "num_format": "0.000E+00",
            "valign": "top"
        }
    )

    url_format = workbook.add_format(
        {
            "font_color": "blue",
            "underline": True,
            "text_wrap": True,
            "valign": "top"
        }
    )

    dataframe_by_sheet = {
        "Sensory_Cache": cache_df,
        "Threshold_Sources": threshold_sources_df,
        "Descriptor_Sources": descriptor_sources_df,
        "Manual_Review": review_df,
        "Input_Compounds": analytes_df,
        "Processing_Summary": summary_df
    }

    for sheet_name, dataframe in dataframe_by_sheet.items():

        worksheet = writer.sheets[sheet_name]

        worksheet.freeze_panes(1, 0)

        if len(dataframe.columns) > 0:

            worksheet.autofilter(
                0,
                0,
                max(len(dataframe), 1),
                len(dataframe.columns) - 1
            )

        # Intestazioni.
        for col_index, column_name in enumerate(
            dataframe.columns
        ):

            worksheet.write(
                0,
                col_index,
                column_name,
                header_format
            )

            # Larghezza stimata.
            if dataframe.empty:
                max_length = len(str(column_name))
            else:
                values_length = (
                    dataframe[column_name]
                    .fillna("")
                    .astype(str)
                    .map(len)
                    .max()
                )

                max_length = max(
                    len(str(column_name)),
                    int(values_length)
                )

            width = min(
                max(max_length + 2, 12),
                45
            )

            worksheet.set_column(
                col_index,
                col_index,
                width,
                wrap_format
            )

        worksheet.set_default_row(30)

    # Formati specifici per il foglio principale.
    if "Sensory_Cache" in writer.sheets:

        sensory_ws = writer.sheets[
            "Sensory_Cache"
        ]

        for column_name in [
            "Odor_threshold_air_value_original",
            "Odor_threshold_air_min_original",
            "Odor_threshold_air_max_original",
            "Odor_threshold_air_ug_m3"
        ]:

            if column_name in cache_df.columns:

                col_index = cache_df.columns.get_loc(
                    column_name
                )

                sensory_ws.set_column(
                    col_index,
                    col_index,
                    18,
                    scientific_format
                )

        for column_name in [
            "Threshold_source_url",
            "Descriptor_source_urls",
            "PubChem_URL"
        ]:

            if column_name in cache_df.columns:

                col_index = cache_df.columns.get_loc(
                    column_name
                )

                sensory_ws.set_column(
                    col_index,
                    col_index,
                    40,
                    url_format
                )


# =====================================================================
# 16. RISULTATO E DOWNLOAD
# =====================================================================

print()
print("=" * 72)
print("ELABORAZIONE COMPLETATA")
print("=" * 72)

print("File creato:", OUTPUT_FILE)
print("Composti elaborati:", len(cache_df))

print(
    "Soglie in aria trovate:",
    int(
        cache_df["Threshold_air_found"]
        .fillna(False)
        .sum()
    )
)

print(
    "Soglie convertite in µg/m³:",
    int(
        cache_df["Odor_threshold_air_ug_m3"]
        .notna()
        .sum()
    )
)

print(
    "Record da controllare manualmente:",
    len(review_df)
)

print()
print("Anteprima dei risultati:")

display_columns = [
    "IUPAC_name_input",
    "CAS",
    "Odor_threshold_air_value_original",
    "Odor_threshold_air_unit_original",
    "Odor_threshold_air_ug_m3",
    "Threshold_type",
    "Odor_descriptors",
    "Flavor_descriptors",
    "Sensory_family",
    "Confidence",
    "Review_required"
]

display(
    cache_df[
        [
            column
            for column in display_columns
            if column in cache_df.columns
        ]
    ]
)

files.download(OUTPUT_FILE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.6 MB/s eta 0:00:00
Chiave API caricata correttamente.

CARICAMENTO DEL FILE
Caricare il file Excel contenente IUPAC_name e CAS.


Saving GCMS_Areas.xlsx to GCMS_Areas.xlsx

File caricato: GCMS_Areas.xlsx
Foglio utilizzato: Compound_Mapping

Composti da elaborare: 4

[1/4] 2-methoxyphenol — 90-05-1
Soglia in aria: 0.084 ng/L_air
Soglia convertita: 0.084 µg/m³
Descrittori olfattivi: smoky; smoked ham-like; vanilla-like; sweet; clove-like; phenolic; spicy; medicinal; woody

[2/4] 3,7-dimethylocta-1,6-dien-3-ol — 78-70-6
Soglia in aria: 3.2 ng/L air
Soglia convertita: 3.2 µg/m³
Descrittori olfattivi: citrus; floral; soapy; fresh; lemon; sweet; woody; lavender; rosewood; petitgrain; bergamot; green; fatty

[3/4] 2,3,5-trimethylpyrazine — 14667-55-1
Soglia in aria: 50.0 ng/L air
Soglia convertita: 50.0 µg/m³
Descrittori olfattivi: roasted; roasty; nutty; hazelnut; peanut; baked potato; cocoa; earthy

[4/4] phenol — 108-95-2
Soglia in aria: 0.011 ppm
Soglia convertita: 42.3398773006135 µg/m³
Descrittori olfattivi: phenolic; medicinal; plastic; rubbery; sweet; tarry

ELABORAZIONE COMPLETATA
File creato: Sensory_Cache.xls

,IUPAC_name_input,CAS,Odor_threshold_air_value_original,Odor_threshold_air_unit_original,Odor_threshold_air_ug_m3,Threshold_type,Odor_descriptors,Flavor_descriptors,Sensory_family,Confidence,Review_required
0,2-methoxyphenol,90-05-1,0.084,ng/L_air,0.084000,unspecified,smoky; smoked ham-like; vanilla-like; sweet; c...,woody; phenolic; bacon; savory; smoky; medicinal,Affumicato | Tostato | Dolce | Speziato | Legn...,high,False
1,"3,7-dimethylocta-1,6-dien-3-ol",78-70-6,3.200,ng/L air,3.200000,Detection,citrus; floral; soapy; fresh; lemon; sweet; wo...,floral; fruity; citrus; bergamot; orange; lemo...,Floreale | Fruttato | Dolce | Verde/Erbaceo | ...,medium,True
2,"2,3,5-trimethylpyrazine",14667-55-1,50.000,ng/L air,50.000000,unspecified,roasted; roasty; nutty; hazelnut; peanut; bake...,toasted; nutty; earthy; chocolate; coffee; coc...,Tostato | Caffè | Frutta secca | Cacao/Cioccol...,medium,True
3,phenol,108-95-2,0.011,ppm,42.339877,Detection,phenolic; medicinal; plastic; rubbery; sweet; ...,sharp; burning,Fenolico/Medicinale | Chimico/Solvente | Dolce,medium,True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =====================================================================
# SECONDA CELLA — COSTRUZIONE DELLA GCMS MASTER TABLE
#
# Questa cella deve essere eseguita nella stessa sessione Colab
# utilizzata per la prima cella.
#
# NON richiede di caricare nuovamente i file.
#
# INPUT già presenti in /content:
#   - file GC-MS contenente i fogli:
#       GCMS_Areas
#       Compound_Mapping
#
#   - Sensory_Cache.xlsx, creato dalla prima cella
#
# OUTPUT:
#   GCMS_Master_Table.xlsx
#
# Non vengono ancora calcolati IPA e LIPA.
# =====================================================================


# =====================================================================
# 0. INSTALLAZIONE DEL SOLO MOTORE EXCEL, SE NECESSARIO
# =====================================================================

!pip -q install xlsxwriter


# =====================================================================
# 1. IMPORTAZIONI
# =====================================================================

import re
from pathlib import Path

import numpy as np
import pandas as pd
import xlsxwriter

from google.colab import files


# =====================================================================
# 2. PARAMETRI
# =====================================================================

WORKING_DIRECTORY = Path("/content")

AREAS_SHEET = "GCMS_Areas"
MAPPING_SHEET = "Compound_Mapping"
SENSORY_SHEET = "Sensory_Cache"

OUTPUT_FILE = WORKING_DIRECTORY / "GCMS_Master_Table.xlsx"
SENSORY_CACHE_FILE = WORKING_DIRECTORY / "Sensory_Cache.xlsx"

SAMPLE_COLUMN = "Sample_ID"
REPLICATE_COLUMN = "Replicate"

MAPPING_NAME_COLUMN = "GCMS_column_name"
IUPAC_COLUMN = "IUPAC_name"
CAS_COLUMN = "CAS"
ROLE_COLUMN = "Compound_role"

THRESHOLD_COLUMN = "Odor_threshold_air_ug_m3"


# =====================================================================
# 3. FUNZIONI DI SUPPORTO
# =====================================================================

def clean_text(value):
    """
    Restituisce una stringa pulita oppure NaN.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text == "" or text.lower() == "nan":
        return np.nan

    return text


def normalize_cas(value):
    """
    Uniforma trattini e spazi nel numero CAS.
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    value = (
        value
        .replace("–", "-")
        .replace("—", "-")
        .replace("−", "-")
    )

    return re.sub(r"\s+", "", value)


def normalize_role(value):
    """
    Uniforma le descrizioni del ruolo del composto.
    """
    if pd.isna(value):
        return ""

    return (
        str(value)
        .strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
    )


def check_required_columns(dataframe, required_columns, table_name):
    """
    Verifica che una tabella contenga tutte le colonne richieste.
    """
    missing = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing:
        raise ValueError(
            f"Nella tabella '{table_name}' mancano le colonne: "
            + ", ".join(missing)
        )


def find_gcms_source_file(directory):
    """
    Cerca nella cartella /content un file Excel contenente
    contemporaneamente i fogli GCMS_Areas e Compound_Mapping.

    Vengono esclusi:
    - Sensory_Cache.xlsx
    - GCMS_Master_Table.xlsx
    - file temporanei Excel
    """

    candidates = []

    for filepath in directory.glob("*.xlsx"):

        if filepath.name.startswith("~$"):
            continue

        if filepath.name in {
            SENSORY_CACHE_FILE.name,
            OUTPUT_FILE.name
        }:
            continue

        try:
            excel_file = pd.ExcelFile(filepath)

            required_sheets = {
                AREAS_SHEET,
                MAPPING_SHEET
            }

            if required_sheets.issubset(
                set(excel_file.sheet_names)
            ):
                candidates.append(filepath)

        except Exception:
            # Il file non è leggibile come workbook Excel valido.
            continue

    if len(candidates) == 0:
        raise FileNotFoundError(
            "Non è stato trovato in /content alcun file Excel "
            f"contenente entrambi i fogli '{AREAS_SHEET}' e "
            f"'{MAPPING_SHEET}'.\n\n"
            "La seconda cella deve essere eseguita nella stessa "
            "sessione Colab della prima cella."
        )

    if len(candidates) == 1:
        return candidates[0]

    # Se esistono più copie, usa quella modificata più recentemente.
    candidates = sorted(
        candidates,
        key=lambda path: path.stat().st_mtime,
        reverse=True
    )

    print(
        "Sono stati trovati più file GC-MS compatibili."
    )
    print(
        "Verrà utilizzato il file modificato più recentemente:"
    )
    print(candidates[0].name)

    print()
    print("Altri file compatibili rilevati:")

    for filepath in candidates[1:]:
        print(" -", filepath.name)

    return candidates[0]


def calculate_column_width(dataframe, column_name, maximum=45):
    """
    Calcola una larghezza leggibile per una colonna Excel.
    """
    header_length = len(str(column_name))

    if dataframe.empty:
        return min(max(header_length + 2, 12), maximum)

    value_length = (
        dataframe[column_name]
        .fillna("")
        .astype(str)
        .map(len)
        .max()
    )

    width = max(
        header_length,
        int(value_length)
    ) + 2

    return min(max(width, 12), maximum)


# =====================================================================
# 4. RICERCA AUTOMATICA DEI FILE GIÀ PRESENTI
# =====================================================================

print("=" * 72)
print("RICERCA DEI FILE NELLA SESSIONE COLAB")
print("=" * 72)

gcms_source_file = find_gcms_source_file(
    WORKING_DIRECTORY
)

if not SENSORY_CACHE_FILE.exists():
    raise FileNotFoundError(
        f"Il file '{SENSORY_CACHE_FILE.name}' non è presente "
        "nella cartella /content.\n\n"
        "Eseguire prima la cella 1 nella stessa sessione Colab "
        "e verificare che abbia creato Sensory_Cache.xlsx."
    )

print("File sorgente GC-MS:", gcms_source_file.name)
print("Cache sensoriale:", SENSORY_CACHE_FILE.name)


# =====================================================================
# 5. CONTROLLO DEI FOGLI
# =====================================================================

gcms_excel = pd.ExcelFile(
    gcms_source_file
)

sensory_excel = pd.ExcelFile(
    SENSORY_CACHE_FILE
)

if AREAS_SHEET not in gcms_excel.sheet_names:
    raise ValueError(
        f"Nel file '{gcms_source_file.name}' manca il foglio "
        f"'{AREAS_SHEET}'."
    )

if MAPPING_SHEET not in gcms_excel.sheet_names:
    raise ValueError(
        f"Nel file '{gcms_source_file.name}' manca il foglio "
        f"'{MAPPING_SHEET}'."
    )

if SENSORY_SHEET not in sensory_excel.sheet_names:
    raise ValueError(
        f"Nel file '{SENSORY_CACHE_FILE.name}' manca il foglio "
        f"'{SENSORY_SHEET}'."
    )

print()
print("Fogli del file GC-MS:", gcms_excel.sheet_names)
print("Fogli della cache:", sensory_excel.sheet_names)


# =====================================================================
# 6. LETTURA DEI DATI
# =====================================================================

areas_df = pd.read_excel(
    gcms_source_file,
    sheet_name=AREAS_SHEET
)

mapping_df = pd.read_excel(
    gcms_source_file,
    sheet_name=MAPPING_SHEET,
    dtype={CAS_COLUMN: str}
)

sensory_df = pd.read_excel(
    SENSORY_CACHE_FILE,
    sheet_name=SENSORY_SHEET,
    dtype={CAS_COLUMN: str}
)

print()
print("Dimensioni matrice delle aree:", areas_df.shape)
print("Dimensioni mapping:", mapping_df.shape)
print("Dimensioni cache sensoriale:", sensory_df.shape)


# =====================================================================
# 7. CONTROLLO DELLE COLONNE OBBLIGATORIE
# =====================================================================

check_required_columns(
    areas_df,
    [
        SAMPLE_COLUMN,
        REPLICATE_COLUMN
    ],
    AREAS_SHEET
)

check_required_columns(
    mapping_df,
    [
        MAPPING_NAME_COLUMN,
        IUPAC_COLUMN,
        CAS_COLUMN,
        ROLE_COLUMN
    ],
    MAPPING_SHEET
)

check_required_columns(
    sensory_df,
    [
        CAS_COLUMN,
        THRESHOLD_COLUMN
    ],
    SENSORY_SHEET
)


# =====================================================================
# 8. PULIZIA DEL MAPPING
# =====================================================================

mapping_df[MAPPING_NAME_COLUMN] = (
    mapping_df[MAPPING_NAME_COLUMN]
    .apply(clean_text)
)

mapping_df[IUPAC_COLUMN] = (
    mapping_df[IUPAC_COLUMN]
    .apply(clean_text)
)

mapping_df[CAS_COLUMN] = (
    mapping_df[CAS_COLUMN]
    .apply(normalize_cas)
)

mapping_df["Compound_role_normalized"] = (
    mapping_df[ROLE_COLUMN]
    .apply(normalize_role)
)


# =====================================================================
# 9. IDENTIFICAZIONE DELLO STANDARD INTERNO
# =====================================================================

internal_standard_labels = {
    "internal standard",
    "internalstandard",
    "standard interno",
    "internal std",
    "is"
}

internal_standard_rows = mapping_df.loc[
    mapping_df["Compound_role_normalized"].isin(
        internal_standard_labels
    )
].copy()

if internal_standard_rows.empty:
    raise ValueError(
        "Nel foglio Compound_Mapping non è stato identificato "
        "alcuno standard interno."
    )

if len(internal_standard_rows) > 1:
    raise ValueError(
        "Nel foglio Compound_Mapping sono presenti più standard "
        "interni. Questa versione gestisce un solo standard interno."
    )

internal_standard_column = internal_standard_rows.iloc[0][
    MAPPING_NAME_COLUMN
]

if internal_standard_column not in areas_df.columns:
    raise ValueError(
        f"La colonna dello standard interno "
        f"'{internal_standard_column}' non è presente nel foglio "
        f"'{AREAS_SHEET}'."
    )

print()
print("Standard interno identificato:", internal_standard_column)


# =====================================================================
# 10. IDENTIFICAZIONE DEGLI ANALITI
# =====================================================================

analyte_labels = {
    "analyte",
    "analita",
    "compound",
    "voc"
}

analyte_mapping = mapping_df.loc[
    mapping_df["Compound_role_normalized"].isin(
        analyte_labels
    )
].copy()

if analyte_mapping.empty:
    raise ValueError(
        "Nel foglio Compound_Mapping non sono stati identificati "
        "analiti."
    )

analyte_mapping = analyte_mapping.drop_duplicates(
    subset=[MAPPING_NAME_COLUMN],
    keep="first"
)

analyte_columns = analyte_mapping[
    MAPPING_NAME_COLUMN
].tolist()

missing_area_columns = [
    column
    for column in analyte_columns
    if column not in areas_df.columns
]

if missing_area_columns:
    raise ValueError(
        "Le seguenti colonne indicate nel mapping non sono presenti "
        "nel foglio GCMS_Areas: "
        + ", ".join(missing_area_columns)
    )

print("Analiti identificati:", len(analyte_columns))

for compound in analyte_columns:
    print(" -", compound)
    # =====================================================================
# 11. CONTROLLO DELLE COLONNE NON MAPPATE
# =====================================================================

metadata_columns = {
    SAMPLE_COLUMN,
    REPLICATE_COLUMN
}

mapped_area_columns = set(
    analyte_columns + [internal_standard_column]
)

unmapped_columns = [
    column
    for column in areas_df.columns
    if column not in metadata_columns
    and column not in mapped_area_columns
]

if unmapped_columns:
    print()
    print(
        "ATTENZIONE: le seguenti colonne non sono presenti nel "
        "Compound_Mapping e non verranno elaborate:"
    )

    for column in unmapped_columns:
        print(" -", column)


# =====================================================================
# 12. CONVERSIONE DELLE AREE IN VALORI NUMERICI
# =====================================================================

numeric_area_columns = (
    analyte_columns
    + [internal_standard_column]
)

for column in numeric_area_columns:

    original_non_empty = (
        areas_df[column]
        .notna()
        .sum()
    )

    areas_df[column] = pd.to_numeric(
        areas_df[column],
        errors="coerce"
    )

    converted_non_empty = (
        areas_df[column]
        .notna()
        .sum()
    )

    if converted_non_empty < original_non_empty:
        print(
            f"Attenzione: nella colonna '{column}' alcuni valori "
            "non numerici sono stati convertiti in dato mancante."
        )


# =====================================================================
# 13. PULIZIA DEGLI IDENTIFICATIVI DEI CAMPIONI
# =====================================================================

areas_df[SAMPLE_COLUMN] = (
    areas_df[SAMPLE_COLUMN]
    .apply(clean_text)
)

areas_df[REPLICATE_COLUMN] = pd.to_numeric(
    areas_df[REPLICATE_COLUMN],
    errors="coerce"
)

if areas_df[SAMPLE_COLUMN].isna().any():
    number_missing = int(
        areas_df[SAMPLE_COLUMN]
        .isna()
        .sum()
    )

    raise ValueError(
        f"Sono presenti {number_missing} righe prive di Sample_ID."
    )

if areas_df[REPLICATE_COLUMN].isna().any():
    print(
        "ATTENZIONE: alcune repliche sono mancanti o non numeriche."
    )


# =====================================================================
# 14. TRASFORMAZIONE DAL FORMATO LARGO AL FORMATO LUNGO
# =====================================================================

long_df = areas_df.melt(
    id_vars=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN,
        internal_standard_column
    ],
    value_vars=analyte_columns,
    var_name=MAPPING_NAME_COLUMN,
    value_name="Area"
)

long_df = long_df.rename(
    columns={
        internal_standard_column: "IS_area"
    }
)

print()
print(
    "Righe create nella tabella lunga:",
    len(long_df)
)


# =====================================================================
# 15. ABBINAMENTO CON IL COMPOUND MAPPING
# =====================================================================

mapping_for_merge = analyte_mapping[
    [
        MAPPING_NAME_COLUMN,
        IUPAC_COLUMN,
        CAS_COLUMN
    ]
].copy()

master_df = long_df.merge(
    mapping_for_merge,
    on=MAPPING_NAME_COLUMN,
    how="left",
    validate="many_to_one"
)

if master_df[CAS_COLUMN].isna().any():

    missing_names = (
        master_df.loc[
            master_df[CAS_COLUMN].isna(),
            MAPPING_NAME_COLUMN
        ]
        .dropna()
        .unique()
        .tolist()
    )

    raise ValueError(
        "Non è stato possibile associare il CAS alle colonne: "
        + ", ".join(missing_names)
    )


# =====================================================================
# 16. CALCOLO DELLE AREE NORMALIZZATE
# =====================================================================

valid_area = (
    master_df["Area"].notna()
    &
    (master_df["Area"] >= 0)
)

valid_internal_standard = (
    master_df["IS_area"].notna()
    &
    (master_df["IS_area"] > 0)
)

master_df["Normalized_area"] = np.where(
    valid_area & valid_internal_standard,
    master_df["Area"] / master_df["IS_area"],
    np.nan
)

master_df["Log10_normalized_area"] = np.where(
    master_df["Normalized_area"] > 0,
    np.log10(master_df["Normalized_area"]),
    np.nan
)


# =====================================================================
# 17. PULIZIA DELLA CACHE SENSORIALE
# =====================================================================

sensory_df[CAS_COLUMN] = (
    sensory_df[CAS_COLUMN]
    .apply(normalize_cas)
)

sensory_df[THRESHOLD_COLUMN] = pd.to_numeric(
    sensory_df[THRESHOLD_COLUMN],
    errors="coerce"
)

duplicated_cas = (
    sensory_df.loc[
        sensory_df[CAS_COLUMN].duplicated(keep=False),
        CAS_COLUMN
    ]
    .dropna()
    .unique()
    .tolist()
)

if duplicated_cas:
    print()
    print(
        "ATTENZIONE: nella cache sensoriale sono presenti "
        "CAS duplicati."
    )
    print(
        "Per ciascun CAS verrà utilizzata la prima riga."
    )

    for cas in duplicated_cas:
        print(" -", cas)

    sensory_df = sensory_df.drop_duplicates(
        subset=[CAS_COLUMN],
        keep="first"
    )


# =====================================================================
# 18. SELEZIONE DEI CAMPI SENSORIALI
# =====================================================================

preferred_sensory_columns = [
    CAS_COLUMN,
    "Common_name",
    "Molecular_formula",
    "Molecular_weight_g_mol",
    "Threshold_air_found",
    "Odor_threshold_air_value_original",
    "Odor_threshold_air_unit_original",
    THRESHOLD_COLUMN,
    "Threshold_conversion_status",
    "Threshold_type",
    "Threshold_conditions",
    "Odor_descriptors",
    "Flavor_descriptors",
    "Sensory_family",
    "Threshold_source_name",
    "Threshold_source_url",
    "Threshold_reference",
    "Descriptor_source_names",
    "Descriptor_source_urls",
    "Confidence",
    "Review_required",
    "Notes",
    "Retrieved_on"
]

sensory_columns_to_merge = [
    column
    for column in preferred_sensory_columns
    if column in sensory_df.columns
]

sensory_for_merge = sensory_df[
    sensory_columns_to_merge
].copy()


# =====================================================================
# 19. ABBINAMENTO TRAMITE CAS
# =====================================================================

master_df = master_df.merge(
    sensory_for_merge,
    on=CAS_COLUMN,
    how="left",
    validate="many_to_one"
)


# =====================================================================
# 20. COLONNE DI CONTROLLO QUALITÀ
# =====================================================================

sensory_record_condition = (
    master_df[THRESHOLD_COLUMN].notna()
)

if "Odor_descriptors" in master_df.columns:
    sensory_record_condition = (
        sensory_record_condition
        |
        master_df["Odor_descriptors"].notna()
    )

master_df["Sensory_record_found"] = np.where(
    sensory_record_condition,
    "Sì",
    "No"
)

master_df["Normalized_area_calculable"] = np.where(
    master_df["Normalized_area"].notna(),
    "Sì",
    "No"
)

master_df["Processing_note"] = ""

master_df.loc[
    master_df["Area"].isna(),
    "Processing_note"
] = "Area analita mancante o non numerica"

master_df.loc[
    master_df["IS_area"].isna(),
    "Processing_note"
] = "Area standard interno mancante o non numerica"

master_df.loc[
    master_df["IS_area"].notna()
    & (master_df["IS_area"] <= 0),
    "Processing_note"
] = "Area standard interno uguale o inferiore a zero"

master_df.loc[
    master_df["Area"] == 0,
    "Processing_note"
] = "Composto non rilevato: area uguale a zero"

master_df.loc[
    master_df[THRESHOLD_COLUMN].isna()
    & (master_df["Processing_note"] == ""),
    "Processing_note"
] = "Soglia olfattiva in aria non disponibile"

master_df.loc[
    master_df[THRESHOLD_COLUMN].notna()
    & (master_df[THRESHOLD_COLUMN] <= 0),
    "Processing_note"
] = "Soglia olfattiva uguale o inferiore a zero"


# =====================================================================
# 21. ORDINAMENTO
# =====================================================================

master_df = master_df.sort_values(
    by=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN,
        MAPPING_NAME_COLUMN
    ],
    ascending=[
        True,
        True,
        True
    ]
).reset_index(drop=True)


# =====================================================================
# 22. ORDINE DELLE COLONNE
# =====================================================================

main_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    MAPPING_NAME_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    "Area",
    "IS_area",
    "Normalized_area",
    "Log10_normalized_area"
]

sensory_output_columns = [
    "Common_name",
    "Molecular_formula",
    "Molecular_weight_g_mol",
    "Threshold_air_found",
    "Odor_threshold_air_value_original",
    "Odor_threshold_air_unit_original",
    THRESHOLD_COLUMN,
    "Threshold_conversion_status",
    "Threshold_type",
    "Threshold_conditions",
    "Odor_descriptors",
    "Flavor_descriptors",
    "Sensory_family",
    "Threshold_source_name",
    "Threshold_source_url",
    "Threshold_reference",
    "Descriptor_source_names",
    "Descriptor_source_urls",
    "Confidence",
    "Review_required",
    "Notes",
    "Retrieved_on"
]

quality_columns = [
    "Sensory_record_found",
    "Normalized_area_calculable",
    "Processing_note"
]

ordered_columns = (
    main_columns
    + [
        column
        for column in sensory_output_columns
        if column in master_df.columns
    ]
    + quality_columns
)

master_df = master_df[
    [
        column
        for column in ordered_columns
        if column in master_df.columns
    ]
]
# =====================================================================
# 23. TABELLA DI RIEPILOGO
# =====================================================================

summary_df = pd.DataFrame(
    {
        "Indicatore": [
            "File GC-MS utilizzato",
            "Cache sensoriale utilizzata",
            "Numero di righe/iniezioni nel file GC-MS",
            "Numero di campioni distinti",
            "Numero di analiti",
            "Numero di righe nella Master Table",
            "Aree normalizzate calcolate",
            "Record con soglia in aria disponibile",
            "Record senza soglia in aria",
            "Standard interno utilizzato"
        ],
        "Valore": [
            gcms_source_file.name,
            SENSORY_CACHE_FILE.name,
            len(areas_df),
            areas_df[SAMPLE_COLUMN].nunique(),
            len(analyte_columns),
            len(master_df),
            int(
                master_df["Normalized_area"]
                .notna()
                .sum()
            ),
            int(
                master_df[THRESHOLD_COLUMN]
                .notna()
                .sum()
            ),
            int(
                master_df[THRESHOLD_COLUMN]
                .isna()
                .sum()
            ),
            internal_standard_column
        ]
    }
)


# =====================================================================
# 24. TABELLA DI REVISIONE
# =====================================================================

review_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    MAPPING_NAME_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    "Area",
    "IS_area",
    "Normalized_area",
    THRESHOLD_COLUMN,
    "Confidence",
    "Review_required",
    "Processing_note"
]

review_columns = [
    column
    for column in review_columns
    if column in master_df.columns
]

review_mask = (
    master_df["Processing_note"] != ""
)

if "Review_required" in master_df.columns:
    review_mask = (
        review_mask
        |
        master_df["Review_required"]
        .fillna(False)
        .astype(bool)
    )

review_df = master_df.loc[
    review_mask,
    review_columns
].copy()


# =====================================================================
# 25. MATRICE DELLE AREE NORMALIZZATE
# =====================================================================

normalized_matrix = master_df.pivot_table(
    index=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN
    ],
    columns=MAPPING_NAME_COLUMN,
    values="Normalized_area",
    aggfunc="first"
)


# =====================================================================
# 26. SALVATAGGIO DEL FILE EXCEL
# =====================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="xlsxwriter"
) as writer:

    master_df.to_excel(
        writer,
        sheet_name="Master_Table",
        index=False
    )

    normalized_matrix.to_excel(
        writer,
        sheet_name="Normalized_Areas"
    )

    summary_df.to_excel(
        writer,
        sheet_name="Processing_Summary",
        index=False
    )

    review_df.to_excel(
        writer,
        sheet_name="Manual_Review",
        index=False
    )

    mapping_df.to_excel(
        writer,
        sheet_name="Mapping_Used",
        index=False
    )

    workbook = writer.book

    header_format = workbook.add_format(
        {
            "bold": True,
            "bg_color": "#D9EAF7",
            "border": 1,
            "text_wrap": True,
            "valign": "top"
        }
    )

    wrap_format = workbook.add_format(
        {
            "text_wrap": True,
            "valign": "top"
        }
    )

    integer_format = workbook.add_format(
        {
            "num_format": "0",
            "valign": "top"
        }
    )

    decimal_format = workbook.add_format(
        {
            "num_format": "0.000000",
            "valign": "top"
        }
    )

    scientific_format = workbook.add_format(
        {
            "num_format": "0.000E+00",
            "valign": "top"
        }
    )

    url_format = workbook.add_format(
        {
            "font_color": "blue",
            "underline": True,
            "text_wrap": True,
            "valign": "top"
        }
    )

    dataframe_by_sheet = {
        "Master_Table": master_df,
        "Processing_Summary": summary_df,
        "Manual_Review": review_df,
        "Mapping_Used": mapping_df
    }

    for sheet_name, dataframe in dataframe_by_sheet.items():

        worksheet = writer.sheets[sheet_name]

        worksheet.freeze_panes(1, 0)
        worksheet.set_default_row(24)

        if len(dataframe.columns) > 0:
            worksheet.autofilter(
                0,
                0,
                max(len(dataframe), 1),
                len(dataframe.columns) - 1
            )

        for column_index, column_name in enumerate(
            dataframe.columns
        ):

            worksheet.write(
                0,
                column_index,
                column_name,
                header_format
            )

            column_width = calculate_column_width(
                dataframe,
                column_name
            )

            worksheet.set_column(
                column_index,
                column_index,
                column_width,
                wrap_format
            )

    master_ws = writer.sheets["Master_Table"]

    for column_name in [
        "Area",
        "IS_area"
    ]:
        if column_name in master_df.columns:

            column_index = master_df.columns.get_loc(
                column_name
            )

            master_ws.set_column(
                column_index,
                column_index,
                14,
                integer_format
            )

    for column_name in [
        "Normalized_area",
        "Log10_normalized_area"
    ]:
        if column_name in master_df.columns:

            column_index = master_df.columns.get_loc(
                column_name
            )

            master_ws.set_column(
                column_index,
                column_index,
                18,
                decimal_format
            )

    if THRESHOLD_COLUMN in master_df.columns:

        column_index = master_df.columns.get_loc(
            THRESHOLD_COLUMN
        )

        master_ws.set_column(
            column_index,
            column_index,
            19,
            scientific_format
        )

    for column_name in [
        "Threshold_source_url",
        "Descriptor_source_urls"
    ]:
        if column_name in master_df.columns:

            column_index = master_df.columns.get_loc(
                column_name
            )

            master_ws.set_column(
                column_index,
                column_index,
                40,
                url_format
            )

    normalized_ws = writer.sheets[
        "Normalized_Areas"
    ]

    normalized_ws.freeze_panes(1, 2)


# =====================================================================
# 27. RISULTATO E DOWNLOAD
# =====================================================================

print()
print("=" * 72)
print("ELABORAZIONE COMPLETATA")
print("=" * 72)

print("File creato:", OUTPUT_FILE.name)
print("File GC-MS utilizzato:", gcms_source_file.name)
print("Cache utilizzata:", SENSORY_CACHE_FILE.name)
print("Righe della Master Table:", len(master_df))
print("Campioni distinti:", areas_df[SAMPLE_COLUMN].nunique())
print("Analiti elaborati:", len(analyte_columns))

print(
    "Aree normalizzate calcolate:",
    int(
        master_df["Normalized_area"]
        .notna()
        .sum()
    )
)

print(
    "Record con soglia in aria:",
    int(
        master_df[THRESHOLD_COLUMN]
        .notna()
        .sum()
    )
)

print(
    "Record da controllare:",
    len(review_df)
)

print()
print("Anteprima della Master Table:")

preview_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    MAPPING_NAME_COLUMN,
    CAS_COLUMN,
    "Area",
    "IS_area",
    "Normalized_area",
    "Log10_normalized_area",
    THRESHOLD_COLUMN,
    "Sensory_family",
    "Confidence",
    "Review_required",
    "Processing_note"
]

preview_columns = [
    column
    for column in preview_columns
    if column in master_df.columns
]

display(
    master_df[
        preview_columns
    ].head(20)
)

files.download(
    str(OUTPUT_FILE)
)

RICERCA DEI FILE NELLA SESSIONE COLAB
File sorgente GC-MS: GCMS_Areas.xlsx
Cache sensoriale: Sensory_Cache.xlsx

Fogli del file GC-MS: ['GCMS_Areas', 'Compound_Mapping', 'metodo_analitico']
Fogli della cache: ['Sensory_Cache', 'Threshold_Sources', 'Descriptor_Sources', 'Manual_Review', 'Input_Compounds', 'Processing_Summary']

Dimensioni matrice delle aree: (6, 7)
Dimensioni mapping: (5, 4)
Dimensioni cache sensoriale: (4, 31)

Standard interno identificato: Internal_standard
Analiti identificati: 4
 - 2-methoxyphenol
 - linalool
 - 2,3,5-trimethylpyrazine
 - phenol

Righe create nella tabella lunga: 24

ELABORAZIONE COMPLETATA
File creato: GCMS_Master_Table.xlsx
File GC-MS utilizzato: GCMS_Areas.xlsx
Cache utilizzata: Sensory_Cache.xlsx
Righe della Master Table: 24
Campioni distinti: 2
Analiti elaborati: 4
Aree normalizzate calcolate: 24
Record con soglia in aria: 24
Record da controllare: 18

Anteprima della Master Table:


,Sample_ID,Replicate,GCMS_column_name,CAS,Area,IS_area,Normalized_area,Log10_normalized_area,Odor_threshold_air_ug_m3,Sensory_family,Confidence,Review_required,Processing_note
0,Cacao_01,1,"2,3,5-trimethylpyrazine",14667-55-1,845230,510000,1.657314,0.219405,50.000000,Tostato | Caffè | Frutta secca | Cacao/Cioccol...,medium,True,
1,Cacao_01,1,2-methoxyphenol,90-05-1,125300,510000,0.245686,-0.609619,0.084000,Affumicato | Tostato | Dolce | Speziato | Legn...,high,False,
2,Cacao_01,1,linalool,78-70-6,64900,510000,0.127255,-0.895325,3.200000,Floreale | Fruttato | Dolce | Verde/Erbaceo | ...,medium,True,
3,Cacao_01,1,phenol,108-95-2,94900,510000,0.186078,-0.730304,42.339877,Fenolico/Medicinale | Chimico/Solvente | Dolce,medium,True,
4,Cacao_01,2,"2,3,5-trimethylpyrazine",14667-55-1,861500,506000,1.702569,0.231105,50.000000,Tostato | Caffè | Frutta secca | Cacao/Cioccol...,medium,True,
5,Cacao_01,2,2-methoxyphenol,90-05-1,128100,506000,0.253162,-0.596601,0.084000,Affumicato | Tostato | Dolce | Speziato | Legn...,high,False,
6,Cacao_01,2,linalool,78-70-6,63200,506000,0.124901,-0.903433,3.200000,Floreale | Fruttato | Dolce | Verde/Erbaceo | ...,medium,True,
7,Cacao_01,2,phenol,108-95-2,93200,506000,0.184190,-0.734735,42.339877,Fenolico/Medicinale | Chimico/Solvente | Dolce,medium,True,
8,Cacao_01,3,"2,3,5-trimethylpyrazine",14667-55-1,837600,514000,1.629572,0.212074,50.000000,Tostato | Caffè | Frutta secca | Cacao/Cioccol...,medium,True,
9,Cacao_01,3,2-methoxyphenol,90-05-1,121900,514000,0.237160,-0.624959,0.084000,Affumicato | Tostato | Dolce | Speziato | Legn...,high,False,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =====================================================================
# TERZA CELLA — CALCOLO DI IPA, LIPA E RANKING
#
# INPUT:
#   - variabile master_df prodotta dalla cella 2
#     oppure
#   - /content/GCMS_Master_Table.xlsx, foglio Master_Table
#
# OUTPUT:
#   /content/GCMS_IPA_Results.xlsx
#
# Nessun nuovo caricamento di file.
# Nessuna chiamata API.
# =====================================================================


# =====================================================================
# 0. INSTALLAZIONE DEL MOTORE EXCEL, SE NECESSARIO
# =====================================================================

!pip -q install xlsxwriter


# =====================================================================
# 1. IMPORTAZIONI
# =====================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import xlsxwriter

from google.colab import files


# =====================================================================
# 2. PARAMETRI
# =====================================================================

WORKING_DIRECTORY = Path("/content")

MASTER_FILE = WORKING_DIRECTORY / "GCMS_Master_Table.xlsx"
MASTER_SHEET = "Master_Table"

OUTPUT_FILE = WORKING_DIRECTORY / "GCMS_IPA_Results.xlsx"

SAMPLE_COLUMN = "Sample_ID"
REPLICATE_COLUMN = "Replicate"

COMPOUND_COLUMN = "GCMS_column_name"
IUPAC_COLUMN = "IUPAC_name"
CAS_COLUMN = "CAS"

NORMALIZED_AREA_COLUMN = "Normalized_area"
THRESHOLD_COLUMN = "Odor_threshold_air_ug_m3"

# Soglia convenzionale di riferimento.
# Deve avere la stessa unità delle soglie della cache.
REFERENCE_THRESHOLD_UG_M3 = 1.0


# =====================================================================
# 3. FUNZIONI DI SUPPORTO
# =====================================================================

def check_required_columns(dataframe, required_columns, table_name):
    """
    Verifica che siano presenti tutte le colonne obbligatorie.
    """
    missing = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing:
        raise ValueError(
            f"Nella tabella '{table_name}' mancano le colonne: "
            + ", ".join(missing)
        )


def calculate_column_width(dataframe, column_name, maximum=45):
    """
    Calcola una larghezza leggibile per una colonna Excel.
    """
    header_length = len(str(column_name))

    if dataframe.empty:
        return min(max(header_length + 2, 12), maximum)

    maximum_value_length = (
        dataframe[column_name]
        .fillna("")
        .astype(str)
        .map(len)
        .max()
    )

    width = max(
        header_length,
        int(maximum_value_length)
    ) + 2

    return min(max(width, 12), maximum)


def classify_lipa(value):
    """
    Classificazione operativa preliminare.

    Non rappresenta una classificazione sensoriale assoluta:
    serve soltanto a facilitare la lettura e il filtraggio.
    """
    if pd.isna(value):
        return "Non calcolabile"

    if value >= 3:
        return "Estremamente elevata"

    if value >= 2:
        return "Molto elevata"

    if value >= 1:
        return "Elevata"

    if value >= 0:
        return "Moderata"

    if value >= -1:
        return "Bassa"

    return "Molto bassa"


def coefficient_of_variation(mean_value, sd_value):
    """
    Calcola CV% evitando divisioni per zero.
    """
    if (
        pd.isna(mean_value)
        or pd.isna(sd_value)
        or mean_value == 0
    ):
        return np.nan

    return 100.0 * sd_value / mean_value


# =====================================================================
# 4. RECUPERO DELLA MASTER TABLE
# =====================================================================

print("=" * 72)
print("RECUPERO DELLA MASTER TABLE")
print("=" * 72)

# Se master_df esiste già nella memoria della sessione,
# ne viene utilizzata una copia.
if (
    "master_df" in globals()
    and isinstance(master_df, pd.DataFrame)
    and not master_df.empty
):

    ipa_df = master_df.copy()

    print(
        "È stata utilizzata la variabile master_df "
        "presente nella memoria della sessione."
    )

else:

    if not MASTER_FILE.exists():
        raise FileNotFoundError(
            f"Non è disponibile la variabile master_df e non è stato "
            f"trovato il file '{MASTER_FILE.name}' in /content.\n\n"
            "Eseguire prima la cella 2 nella stessa sessione Colab."
        )

    master_excel = pd.ExcelFile(
        MASTER_FILE
    )

    if MASTER_SHEET not in master_excel.sheet_names:
        raise ValueError(
            f"Nel file '{MASTER_FILE.name}' manca il foglio "
            f"'{MASTER_SHEET}'."
        )

    ipa_df = pd.read_excel(
        MASTER_FILE,
        sheet_name=MASTER_SHEET,
        dtype={CAS_COLUMN: str}
    )

    print(
        f"È stato letto il file '{MASTER_FILE.name}', "
        f"foglio '{MASTER_SHEET}'."
    )

print("Righe disponibili:", len(ipa_df))


# =====================================================================
# 5. CONTROLLO DELLE COLONNE OBBLIGATORIE
# =====================================================================

required_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    COMPOUND_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    "Area",
    "IS_area",
    NORMALIZED_AREA_COLUMN,
    THRESHOLD_COLUMN
]

check_required_columns(
    ipa_df,
    required_columns,
    MASTER_SHEET
)


# =====================================================================
# 6. CONVERSIONE DELLE COLONNE NUMERICHE
# =====================================================================

numeric_columns = [
    "Area",
    "IS_area",
    NORMALIZED_AREA_COLUMN,
    THRESHOLD_COLUMN
]

for column in numeric_columns:
    ipa_df[column] = pd.to_numeric(
        ipa_df[column],
        errors="coerce"
    )

ipa_df[REPLICATE_COLUMN] = pd.to_numeric(
    ipa_df[REPLICATE_COLUMN],
    errors="coerce"
)


# =====================================================================
# 7. CONTROLLO DELLA SOGLIA DI RIFERIMENTO
# =====================================================================

if REFERENCE_THRESHOLD_UG_M3 <= 0:
    raise ValueError(
        "REFERENCE_THRESHOLD_UG_M3 deve essere maggiore di zero."
    )


# =====================================================================
# 8. VALIDITÀ DEI DATI PER IL CALCOLO
# =====================================================================

valid_normalized_area = (
    ipa_df[NORMALIZED_AREA_COLUMN].notna()
    &
    (ipa_df[NORMALIZED_AREA_COLUMN] > 0)
)

valid_threshold = (
    ipa_df[THRESHOLD_COLUMN].notna()
    &
    (ipa_df[THRESHOLD_COLUMN] > 0)
)

ipa_df["IPA_calculable"] = np.where(
    valid_normalized_area & valid_threshold,
    "Sì",
    "No"
)


# =====================================================================
# 9. CALCOLO DI IPA
# =====================================================================

ipa_df["IPA"] = np.where(
    valid_normalized_area & valid_threshold,

    ipa_df[NORMALIZED_AREA_COLUMN]
    * REFERENCE_THRESHOLD_UG_M3
    / ipa_df[THRESHOLD_COLUMN],

    np.nan
)


# =====================================================================
# 10. CALCOLO DI LIPA
# =====================================================================

ipa_df["LIPA"] = np.where(
    ipa_df["IPA"] > 0,
    np.log10(ipa_df["IPA"]),
    np.nan
)


# =====================================================================
# 11. SCOMPOSIZIONE DEL LIPA
#
# LIPA = contributo analitico + contributo della soglia
# =====================================================================

ipa_df["Log10_analytical_response"] = np.where(
    ipa_df[NORMALIZED_AREA_COLUMN] > 0,
    np.log10(
        ipa_df[NORMALIZED_AREA_COLUMN]
    ),
    np.nan
)

ipa_df["Log10_odor_potency"] = np.where(
    ipa_df[THRESHOLD_COLUMN] > 0,

    np.log10(
        REFERENCE_THRESHOLD_UG_M3
        / ipa_df[THRESHOLD_COLUMN]
    ),

    np.nan
)


# =====================================================================
# 12. CLASSE OPERATIVA
# =====================================================================

ipa_df["IPA_priority_class"] = (
    ipa_df["LIPA"].apply(classify_lipa)
)


# =====================================================================
# 13. NOTE SUL CALCOLO
# =====================================================================

ipa_df["IPA_processing_note"] = ""

ipa_df.loc[
    ipa_df[NORMALIZED_AREA_COLUMN].isna(),
    "IPA_processing_note"
] = "Area normalizzata non disponibile"

ipa_df.loc[
    ipa_df[NORMALIZED_AREA_COLUMN].notna()
    & (ipa_df[NORMALIZED_AREA_COLUMN] <= 0),
    "IPA_processing_note"
] = "Area normalizzata uguale o inferiore a zero"

ipa_df.loc[
    ipa_df[THRESHOLD_COLUMN].isna(),
    "IPA_processing_note"
] = "Soglia olfattiva in aria non disponibile"

ipa_df.loc[
    ipa_df[THRESHOLD_COLUMN].notna()
    & (ipa_df[THRESHOLD_COLUMN] <= 0),
    "IPA_processing_note"
] = "Soglia olfattiva uguale o inferiore a zero"


# =====================================================================
# 14. RANKING ENTRO CIASCUNA REPLICA
# =====================================================================

ranking_group_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN
]

ipa_df["IPA_rank_replica"] = (
    ipa_df
    .groupby(
        ranking_group_columns,
        dropna=False
    )["LIPA"]
    .rank(
        method="dense",
        ascending=False,
        na_option="bottom"
    )
)

ipa_df["IPA_rank_replica"] = (
    ipa_df["IPA_rank_replica"]
    .astype("Int64")
)


# =====================================================================
# 15. ORDINAMENTO DELLA TABELLA A LIVELLO DI REPLICA
# =====================================================================

ipa_df = ipa_df.sort_values(
    by=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN,
        "IPA_rank_replica",
        COMPOUND_COLUMN
    ],
    ascending=[
        True,
        True,
        True,
        True
    ],
    na_position="last"
).reset_index(drop=True)


# =====================================================================
# 16. RIEPILOGO PER CAMPIONE
#
# Le aree normalizzate vengono mediate tra le repliche.
# L'IPA del campione viene calcolato sulla media delle aree normalizzate.
# =====================================================================

sample_group_columns = [
    SAMPLE_COLUMN,
    COMPOUND_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN
]

aggregation_dictionary = {
    NORMALIZED_AREA_COLUMN: [
        "count",
        "mean",
        "std",
        "min",
        "max"
    ],
    THRESHOLD_COLUMN: "first"
}

# Conserva anche i principali dati sensoriali, se presenti.
optional_first_columns = [
    "Common_name",
    "Threshold_type",
    "Odor_descriptors",
    "Flavor_descriptors",
    "Sensory_family",
    "Confidence",
    "Review_required",
    "Threshold_source_name",
    "Threshold_source_url"
]

for column in optional_first_columns:
    if column in ipa_df.columns:
        aggregation_dictionary[column] = "first"

sample_summary = (
    ipa_df
    .groupby(
        sample_group_columns,
        dropna=False
    )
    .agg(aggregation_dictionary)
    .reset_index()
)


# =====================================================================
# 17. APPIATTIMENTO DELLE INTESTAZIONI DEL RIEPILOGO
# =====================================================================

flattened_columns = []

for column in sample_summary.columns:

    if isinstance(column, tuple):

        first_part = str(column[0]).strip()
        second_part = str(column[1]).strip()

        if second_part:
            flattened_columns.append(
                f"{first_part}_{second_part}"
            )
        else:
            flattened_columns.append(first_part)

    else:
        flattened_columns.append(str(column))

sample_summary.columns = flattened_columns


# Rinomina le colonne principali.
rename_summary_columns = {
    f"{NORMALIZED_AREA_COLUMN}_count": "Number_of_replicates",
    f"{NORMALIZED_AREA_COLUMN}_mean": "Mean_normalized_area",
    f"{NORMALIZED_AREA_COLUMN}_std": "SD_normalized_area",
    f"{NORMALIZED_AREA_COLUMN}_min": "Min_normalized_area",
    f"{NORMALIZED_AREA_COLUMN}_max": "Max_normalized_area",
    f"{THRESHOLD_COLUMN}_first": THRESHOLD_COLUMN
}

for column in optional_first_columns:
    rename_summary_columns[
        f"{column}_first"
    ] = column

sample_summary = sample_summary.rename(
    columns=rename_summary_columns
)


# =====================================================================
# 18. COEFFICIENTE DI VARIAZIONE DELLE REPLICHE
# =====================================================================

sample_summary["CV_normalized_area_percent"] = (
    sample_summary.apply(
        lambda row: coefficient_of_variation(
            row["Mean_normalized_area"],
            row["SD_normalized_area"]
        ),
        axis=1
    )
)


# =====================================================================
# 19. IPA E LIPA SULLA MEDIA DELLE REPLICHE
# =====================================================================

valid_sample_mean = (
    sample_summary["Mean_normalized_area"].notna()
    &
    (sample_summary["Mean_normalized_area"] > 0)
)

valid_sample_threshold = (
    sample_summary[THRESHOLD_COLUMN].notna()
    &
    (sample_summary[THRESHOLD_COLUMN] > 0)
)

sample_summary["IPA_mean"] = np.where(
    valid_sample_mean & valid_sample_threshold,

    sample_summary["Mean_normalized_area"]
    * REFERENCE_THRESHOLD_UG_M3
    / sample_summary[THRESHOLD_COLUMN],

    np.nan
)

sample_summary["LIPA_mean"] = np.where(
    sample_summary["IPA_mean"] > 0,
    np.log10(
        sample_summary["IPA_mean"]
    ),
    np.nan
)

sample_summary["IPA_priority_class"] = (
    sample_summary["LIPA_mean"]
    .apply(classify_lipa)
)


# =====================================================================
# 20. RANKING ENTRO CIASCUN CAMPIONE
# =====================================================================

sample_summary["IPA_rank_sample"] = (
    sample_summary
    .groupby(
        SAMPLE_COLUMN,
        dropna=False
    )["LIPA_mean"]
    .rank(
        method="dense",
        ascending=False,
        na_option="bottom"
    )
)

sample_summary["IPA_rank_sample"] = (
    sample_summary["IPA_rank_sample"]
    .astype("Int64")
)


sample_summary = sample_summary.sort_values(
    by=[
        SAMPLE_COLUMN,
        "IPA_rank_sample",
        COMPOUND_COLUMN
    ],
    ascending=[
        True,
        True,
        True
    ],
    na_position="last"
).reset_index(drop=True)


# =====================================================================
# 21. MATRICE IPA PER REPLICA
# =====================================================================

ipa_matrix_replica = ipa_df.pivot_table(
    index=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN
    ],
    columns=COMPOUND_COLUMN,
    values="IPA",
    aggfunc="first"
)


# =====================================================================
# 22. MATRICE LIPA PER REPLICA
# =====================================================================

lipa_matrix_replica = ipa_df.pivot_table(
    index=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN
    ],
    columns=COMPOUND_COLUMN,
    values="LIPA",
    aggfunc="first"
)


# =====================================================================
# 23. MATRICE IPA SULLA MEDIA DEL CAMPIONE
# =====================================================================

ipa_matrix_sample = sample_summary.pivot_table(
    index=SAMPLE_COLUMN,
    columns=COMPOUND_COLUMN,
    values="IPA_mean",
    aggfunc="first"
)


# =====================================================================
# 24. MATRICE LIPA SULLA MEDIA DEL CAMPIONE
# =====================================================================

lipa_matrix_sample = sample_summary.pivot_table(
    index=SAMPLE_COLUMN,
    columns=COMPOUND_COLUMN,
    values="LIPA_mean",
    aggfunc="first"
)


# =====================================================================
# 25. TOP COMPOUNDS PER CAMPIONE
# =====================================================================

TOP_N = 20

top_compounds = sample_summary.loc[
    sample_summary["IPA_rank_sample"].notna()
    &
    (
        sample_summary["IPA_rank_sample"]
        <= TOP_N
    )
].copy()


# =====================================================================
# 26. RECORD NON CALCOLABILI
# =====================================================================

not_calculable = ipa_df.loc[
    ipa_df["IPA"].isna()
].copy()

not_calculable_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    COMPOUND_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    NORMALIZED_AREA_COLUMN,
    THRESHOLD_COLUMN,
    "IPA_processing_note"
]

not_calculable_columns = [
    column
    for column in not_calculable_columns
    if column in not_calculable.columns
]

not_calculable = not_calculable[
    not_calculable_columns
]


# =====================================================================
# 27. RIEPILOGO DELL'ELABORAZIONE
# =====================================================================

processing_summary = pd.DataFrame(
    {
        "Indicatore": [
            "Soglia di riferimento T_r (µg/m³)",
            "Numero di righe analita-replica",
            "Numero di campioni distinti",
            "Numero di analiti distinti",
            "Valori IPA calcolati",
            "Valori IPA non calcolabili",
            "Numero di righe nel riepilogo campione",
            "Top N utilizzato"
        ],
        "Valore": [
            REFERENCE_THRESHOLD_UG_M3,
            len(ipa_df),
            ipa_df[SAMPLE_COLUMN].nunique(),
            ipa_df[CAS_COLUMN].nunique(),
            int(
                ipa_df["IPA"]
                .notna()
                .sum()
            ),
            int(
                ipa_df["IPA"]
                .isna()
                .sum()
            ),
            len(sample_summary),
            TOP_N
        ]
    }
)


# =====================================================================
# 28. ORDINE DELLE COLONNE DEL RISULTATO A LIVELLO DI REPLICA
# =====================================================================

main_replica_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    "IPA_rank_replica",
    COMPOUND_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    "Area",
    "IS_area",
    NORMALIZED_AREA_COLUMN,
    THRESHOLD_COLUMN,
    "IPA",
    "LIPA",
    "Log10_analytical_response",
    "Log10_odor_potency",
    "IPA_priority_class"
]

sensory_replica_columns = [
    "Threshold_type",
    "Odor_descriptors",
    "Flavor_descriptors",
    "Sensory_family",
    "Confidence",
    "Review_required",
    "Threshold_source_name",
    "Threshold_source_url"
]

quality_replica_columns = [
    "IPA_calculable",
    "IPA_processing_note"
]

ordered_replica_columns = (
    main_replica_columns
    + [
        column
        for column in sensory_replica_columns
        if column in ipa_df.columns
    ]
    + quality_replica_columns
)

# Conserva alla fine eventuali colonne originali non incluse sopra.
remaining_columns = [
    column
    for column in ipa_df.columns
    if column not in ordered_replica_columns
]

ipa_df = ipa_df[
    ordered_replica_columns
    + remaining_columns
]


# =====================================================================
# 29. SALVATAGGIO DEL FILE EXCEL
# =====================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="xlsxwriter"
) as writer:

    ipa_df.to_excel(
        writer,
        sheet_name="IPA_by_Replicate",
        index=False
    )

    sample_summary.to_excel(
        writer,
        sheet_name="IPA_by_Sample",
        index=False
    )

    top_compounds.to_excel(
        writer,
        sheet_name="Top_Compounds",
        index=False
    )

    ipa_matrix_replica.to_excel(
        writer,
        sheet_name="IPA_Matrix_Replicate"
    )

    lipa_matrix_replica.to_excel(
        writer,
        sheet_name="LIPA_Matrix_Replicate"
    )

    ipa_matrix_sample.to_excel(
        writer,
        sheet_name="IPA_Matrix_Sample"
    )

    lipa_matrix_sample.to_excel(
        writer,
        sheet_name="LIPA_Matrix_Sample"
    )

    not_calculable.to_excel(
        writer,
        sheet_name="IPA_Not_Calculable",
        index=False
    )

    processing_summary.to_excel(
        writer,
        sheet_name="Processing_Summary",
        index=False
    )

    # -----------------------------------------------------------------
    # Formati Excel
    # -----------------------------------------------------------------

    workbook = writer.book

    header_format = workbook.add_format(
        {
            "bold": True,
            "bg_color": "#D9EAF7",
            "border": 1,
            "text_wrap": True,
            "valign": "top"
        }
    )

    wrap_format = workbook.add_format(
        {
            "text_wrap": True,
            "valign": "top"
        }
    )

    integer_format = workbook.add_format(
        {
            "num_format": "0",
            "valign": "top"
        }
    )

    decimal_format = workbook.add_format(
        {
            "num_format": "0.000000",
            "valign": "top"
        }
    )

    scientific_format = workbook.add_format(
        {
            "num_format": "0.000E+00",
            "valign": "top"
        }
    )

    rank_format = workbook.add_format(
        {
            "num_format": "0",
            "align": "center",
            "valign": "top"
        }
    )

    url_format = workbook.add_format(
        {
            "font_color": "blue",
            "underline": True,
            "text_wrap": True,
            "valign": "top"
        }
    )

    # Colori usati soltanto per la formattazione condizionale.
    high_format = workbook.add_format(
        {
            "bg_color": "#C6EFCE",
            "font_color": "#006100"
        }
    )

    medium_format = workbook.add_format(
        {
            "bg_color": "#FFEB9C",
            "font_color": "#9C6500"
        }
    )

    low_format = workbook.add_format(
        {
            "bg_color": "#FFC7CE",
            "font_color": "#9C0006"
        }
    )


    dataframe_by_sheet = {
        "IPA_by_Replicate": ipa_df,
        "IPA_by_Sample": sample_summary,
        "Top_Compounds": top_compounds,
        "IPA_Not_Calculable": not_calculable,
        "Processing_Summary": processing_summary
    }

    for sheet_name, dataframe in dataframe_by_sheet.items():

        worksheet = writer.sheets[sheet_name]

        worksheet.freeze_panes(1, 0)
        worksheet.set_default_row(24)

        if len(dataframe.columns) > 0:

            worksheet.autofilter(
                0,
                0,
                max(len(dataframe), 1),
                len(dataframe.columns) - 1
            )

        for column_index, column_name in enumerate(
            dataframe.columns
        ):

            worksheet.write(
                0,
                column_index,
                column_name,
                header_format
            )

            width = calculate_column_width(
                dataframe,
                column_name
            )

            worksheet.set_column(
                column_index,
                column_index,
                width,
                wrap_format
            )


    # -----------------------------------------------------------------
    # Formattazione IPA_by_Replicate
    # -----------------------------------------------------------------

    replica_ws = writer.sheets[
        "IPA_by_Replicate"
    ]

    for column_name in [
        "Area",
        "IS_area"
    ]:

        if column_name in ipa_df.columns:

            column_index = ipa_df.columns.get_loc(
                column_name
            )

            replica_ws.set_column(
                column_index,
                column_index,
                14,
                integer_format
            )

    for column_name in [
        NORMALIZED_AREA_COLUMN,
        THRESHOLD_COLUMN,
        "IPA"
    ]:

        if column_name in ipa_df.columns:

            column_index = ipa_df.columns.get_loc(
                column_name
            )

            replica_ws.set_column(
                column_index,
                column_index,
                18,
                scientific_format
            )

    for column_name in [
        "LIPA",
        "Log10_analytical_response",
        "Log10_odor_potency"
    ]:

        if column_name in ipa_df.columns:

            column_index = ipa_df.columns.get_loc(
                column_name
            )

            replica_ws.set_column(
                column_index,
                column_index,
                18,
                decimal_format
            )

    if "IPA_rank_replica" in ipa_df.columns:

        column_index = ipa_df.columns.get_loc(
            "IPA_rank_replica"
        )

        replica_ws.set_column(
            column_index,
            column_index,
            12,
            rank_format
        )

    if "Threshold_source_url" in ipa_df.columns:

        column_index = ipa_df.columns.get_loc(
            "Threshold_source_url"
        )

        replica_ws.set_column(
            column_index,
            column_index,
            40,
            url_format
        )

    if "LIPA" in ipa_df.columns and len(ipa_df) > 0:

        lipa_col = ipa_df.columns.get_loc("LIPA")

        replica_ws.conditional_format(
            1,
            lipa_col,
            len(ipa_df),
            lipa_col,
            {
                "type": "3_color_scale",
                "min_color": "#F8696B",
                "mid_color": "#FFEB84",
                "max_color": "#63BE7B"
            }
        )


    # -----------------------------------------------------------------
    # Formattazione IPA_by_Sample
    # -----------------------------------------------------------------

    sample_ws = writer.sheets[
        "IPA_by_Sample"
    ]

    for column_name in [
        "Mean_normalized_area",
        "SD_normalized_area",
        "Min_normalized_area",
        "Max_normalized_area",
        THRESHOLD_COLUMN,
        "IPA_mean"
    ]:

        if column_name in sample_summary.columns:

            column_index = (
                sample_summary.columns.get_loc(
                    column_name
                )
            )

            sample_ws.set_column(
                column_index,
                column_index,
                18,
                scientific_format
            )

    for column_name in [
        "LIPA_mean",
        "CV_normalized_area_percent"
    ]:

        if column_name in sample_summary.columns:

            column_index = (
                sample_summary.columns.get_loc(
                    column_name
                )
            )

            sample_ws.set_column(
                column_index,
                column_index,
                18,
                decimal_format
            )

    if "IPA_rank_sample" in sample_summary.columns:

        column_index = (
            sample_summary.columns.get_loc(
                "IPA_rank_sample"
            )
        )

        sample_ws.set_column(
            column_index,
            column_index,
            12,
            rank_format
        )

    if "LIPA_mean" in sample_summary.columns and len(sample_summary) > 0:

        lipa_mean_col = (
            sample_summary.columns.get_loc(
                "LIPA_mean"
            )
        )

        sample_ws.conditional_format(
            1,
            lipa_mean_col,
            len(sample_summary),
            lipa_mean_col,
            {
                "type": "3_color_scale",
                "min_color": "#F8696B",
                "mid_color": "#FFEB84",
                "max_color": "#63BE7B"
            }
        )


    # -----------------------------------------------------------------
    # Formattazione delle matrici
    # -----------------------------------------------------------------

    matrix_sheet_names = [
        "IPA_Matrix_Replicate",
        "LIPA_Matrix_Replicate",
        "IPA_Matrix_Sample",
        "LIPA_Matrix_Sample"
    ]

    for sheet_name in matrix_sheet_names:

        worksheet = writer.sheets[sheet_name]

        worksheet.freeze_panes(1, 2)

        worksheet.set_column(
            0,
            1,
            18
        )

        worksheet.set_column(
            2,
            200,
            16,
            scientific_format
            if sheet_name.startswith("IPA_")
            else decimal_format
        )


# =====================================================================
# 30. RISULTATO E DOWNLOAD
# =====================================================================

print()
print("=" * 72)
print("CALCOLO IPA COMPLETATO")
print("=" * 72)

print("File creato:", OUTPUT_FILE.name)

print(
    "Righe analita-replica:",
    len(ipa_df)
)

print(
    "Valori IPA calcolati:",
    int(
        ipa_df["IPA"]
        .notna()
        .sum()
    )
)

print(
    "Valori IPA non calcolabili:",
    int(
        ipa_df["IPA"]
        .isna()
        .sum()
    )
)

print(
    "Campioni distinti:",
    ipa_df[SAMPLE_COLUMN].nunique()
)

print(
    "Analiti distinti:",
    ipa_df[CAS_COLUMN].nunique()
)

print()
print("Anteprima del ranking medio per campione:")

preview_columns = [
    SAMPLE_COLUMN,
    "IPA_rank_sample",
    COMPOUND_COLUMN,
    CAS_COLUMN,
    "Mean_normalized_area",
    THRESHOLD_COLUMN,
    "IPA_mean",
    "LIPA_mean",
    "IPA_priority_class",
    "Sensory_family",
    "Confidence"
]

preview_columns = [
    column
    for column in preview_columns
    if column in sample_summary.columns
]

display(
    sample_summary[
        preview_columns
    ].head(30)
)

files.download(
    str(OUTPUT_FILE)
)

RECUPERO DELLA MASTER TABLE
È stata utilizzata la variabile master_df presente nella memoria della sessione.
Righe disponibili: 24

CALCOLO IPA COMPLETATO
File creato: GCMS_IPA_Results.xlsx
Righe analita-replica: 24
Valori IPA calcolati: 24
Valori IPA non calcolabili: 0
Campioni distinti: 2
Analiti distinti: 4

Anteprima del ranking medio per campione:


,Sample_ID,IPA_rank_sample,GCMS_column_name,CAS,Mean_normalized_area,Odor_threshold_air_ug_m3,IPA_mean,LIPA_mean,IPA_priority_class,Sensory_family,Confidence
0,Cacao_01,1,2-methoxyphenol,90-05-1,0.245336,0.084000,2.920666,0.465482,Moderata,Affumicato | Tostato | Dolce | Speziato | Legn...,high
1,Cacao_01,2,linalool,78-70-6,0.126724,3.200000,0.039601,-1.402292,Molto bassa,Floreale | Fruttato | Dolce | Verde/Erbaceo | ...,medium
2,Cacao_01,3,"2,3,5-trimethylpyrazine",14667-55-1,1.663152,50.000000,0.033263,-1.478038,Molto bassa,Tostato | Caffè | Frutta secca | Cacao/Cioccol...,medium
3,Cacao_01,4,phenol,108-95-2,0.185550,42.339877,0.004382,-2.358289,Molto bassa,Fenolico/Medicinale | Chimico/Solvente | Dolce,medium
4,Cacao_02,1,2-methoxyphenol,90-05-1,0.199480,0.084000,2.374760,0.375620,Moderata,Affumicato | Tostato | Dolce | Speziato | Legn...,high
5,Cacao_02,2,linalool,78-70-6,0.145738,3.200000,0.045543,-1.341576,Molto bassa,Floreale | Fruttato | Dolce | Verde/Erbaceo | ...,medium
6,Cacao_02,3,"2,3,5-trimethylpyrazine",14667-55-1,1.845060,50.000000,0.036901,-1.432959,Molto bassa,Tostato | Caffè | Frutta secca | Cacao/Cioccol...,medium
7,Cacao_02,4,phenol,108-95-2,0.253445,42.339877,0.005986,-2.222867,Molto bassa,Fenolico/Medicinale | Chimico/Solvente | Dolce,medium


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =====================================================================
# SECONDA CELLA — COSTRUZIONE DELLA GCMS MASTER TABLE
#
# Questa cella deve essere eseguita nella stessa sessione Colab
# utilizzata per la prima cella.
#
# NON richiede di caricare nuovamente i file.
#
# INPUT già presenti in /content:
#   - file GC-MS contenente i fogli:
#       GCMS_Areas
#       Compound_Mapping
#
#   - Sensory_Cache.xlsx, creato dalla prima cella
#
# OUTPUT:
#   GCMS_Master_Table.xlsx
#
# Non vengono ancora calcolati IPA e LIPA.
# =====================================================================


# =====================================================================
# 0. INSTALLAZIONE DEL SOLO MOTORE EXCEL, SE NECESSARIO
# =====================================================================

!pip -q install xlsxwriter


# =====================================================================
# 1. IMPORTAZIONI
# =====================================================================

import re
from pathlib import Path

import numpy as np
import pandas as pd
import xlsxwriter

from google.colab import files


# =====================================================================
# 2. PARAMETRI
# =====================================================================

WORKING_DIRECTORY = Path("/content")

AREAS_SHEET = "GCMS_Areas"
MAPPING_SHEET = "Compound_Mapping"
SENSORY_SHEET = "Sensory_Cache"

OUTPUT_FILE = WORKING_DIRECTORY / "GCMS_Master_Table.xlsx"
SENSORY_CACHE_FILE = WORKING_DIRECTORY / "Sensory_Cache.xlsx"

SAMPLE_COLUMN = "Sample_ID"
REPLICATE_COLUMN = "Replicate"

MAPPING_NAME_COLUMN = "GCMS_column_name"
IUPAC_COLUMN = "IUPAC_name"
CAS_COLUMN = "CAS"
ROLE_COLUMN = "Compound_role"

THRESHOLD_COLUMN = "Odor_threshold_air_ug_m3"


# =====================================================================
# 3. FUNZIONI DI SUPPORTO
# =====================================================================

def clean_text(value):
    """
    Restituisce una stringa pulita oppure NaN.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text == "" or text.lower() == "nan":
        return np.nan

    return text


def normalize_cas(value):
    """
    Uniforma trattini e spazi nel numero CAS.
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    value = (
        value
        .replace("–", "-")
        .replace("—", "-")
        .replace("−", "-")
    )

    return re.sub(r"\s+", "", value)


def normalize_role(value):
    """
    Uniforma le descrizioni del ruolo del composto.
    """
    if pd.isna(value):
        return ""

    return (
        str(value)
        .strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
    )


def check_required_columns(dataframe, required_columns, table_name):
    """
    Verifica che una tabella contenga tutte le colonne richieste.
    """
    missing = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing:
        raise ValueError(
            f"Nella tabella '{table_name}' mancano le colonne: "
            + ", ".join(missing)
        )


def find_gcms_source_file(directory):
    """
    Cerca nella cartella /content un file Excel contenente
    contemporaneamente i fogli GCMS_Areas e Compound_Mapping.

    Vengono esclusi:
    - Sensory_Cache.xlsx
    - GCMS_Master_Table.xlsx
    - file temporanei Excel
    """

    candidates = []

    for filepath in directory.glob("*.xlsx"):

        if filepath.name.startswith("~$"):
            continue

        if filepath.name in {
            SENSORY_CACHE_FILE.name,
            OUTPUT_FILE.name
        }:
            continue

        try:
            excel_file = pd.ExcelFile(filepath)

            required_sheets = {
                AREAS_SHEET,
                MAPPING_SHEET
            }

            if required_sheets.issubset(
                set(excel_file.sheet_names)
            ):
                candidates.append(filepath)

        except Exception:
            # Il file non è leggibile come workbook Excel valido.
            continue

    if len(candidates) == 0:
        raise FileNotFoundError(
            "Non è stato trovato in /content alcun file Excel "
            f"contenente entrambi i fogli '{AREAS_SHEET}' e "
            f"'{MAPPING_SHEET}'.\n\n"
            "La seconda cella deve essere eseguita nella stessa "
            "sessione Colab della prima cella."
        )

    if len(candidates) == 1:
        return candidates[0]

    # Se esistono più copie, usa quella modificata più recentemente.
    candidates = sorted(
        candidates,
        key=lambda path: path.stat().st_mtime,
        reverse=True
    )

    print(
        "Sono stati trovati più file GC-MS compatibili."
    )
    print(
        "Verrà utilizzato il file modificato più recentemente:"
    )
    print(candidates[0].name)

    print()
    print("Altri file compatibili rilevati:")

    for filepath in candidates[1:]:
        print(" -", filepath.name)

    return candidates[0]


def calculate_column_width(dataframe, column_name, maximum=45):
    """
    Calcola una larghezza leggibile per una colonna Excel.
    """
    header_length = len(str(column_name))

    if dataframe.empty:
        return min(max(header_length + 2, 12), maximum)

    value_length = (
        dataframe[column_name]
        .fillna("")
        .astype(str)
        .map(len)
        .max()
    )

    width = max(
        header_length,
        int(value_length)
    ) + 2

    return min(max(width, 12), maximum)


# =====================================================================
# 4. RICERCA AUTOMATICA DEI FILE GIÀ PRESENTI
# =====================================================================

print("=" * 72)
print("RICERCA DEI FILE NELLA SESSIONE COLAB")
print("=" * 72)

gcms_source_file = find_gcms_source_file(
    WORKING_DIRECTORY
)

if not SENSORY_CACHE_FILE.exists():
    raise FileNotFoundError(
        f"Il file '{SENSORY_CACHE_FILE.name}' non è presente "
        "nella cartella /content.\n\n"
        "Eseguire prima la cella 1 nella stessa sessione Colab "
        "e verificare che abbia creato Sensory_Cache.xlsx."
    )

print("File sorgente GC-MS:", gcms_source_file.name)
print("Cache sensoriale:", SENSORY_CACHE_FILE.name)


# =====================================================================
# 5. CONTROLLO DEI FOGLI
# =====================================================================

gcms_excel = pd.ExcelFile(
    gcms_source_file
)

sensory_excel = pd.ExcelFile(
    SENSORY_CACHE_FILE
)

if AREAS_SHEET not in gcms_excel.sheet_names:
    raise ValueError(
        f"Nel file '{gcms_source_file.name}' manca il foglio "
        f"'{AREAS_SHEET}'."
    )

if MAPPING_SHEET not in gcms_excel.sheet_names:
    raise ValueError(
        f"Nel file '{gcms_source_file.name}' manca il foglio "
        f"'{MAPPING_SHEET}'."
    )

if SENSORY_SHEET not in sensory_excel.sheet_names:
    raise ValueError(
        f"Nel file '{SENSORY_CACHE_FILE.name}' manca il foglio "
        f"'{SENSORY_SHEET}'."
    )

print()
print("Fogli del file GC-MS:", gcms_excel.sheet_names)
print("Fogli della cache:", sensory_excel.sheet_names)


# =====================================================================
# 6. LETTURA DEI DATI
# =====================================================================

areas_df = pd.read_excel(
    gcms_source_file,
    sheet_name=AREAS_SHEET
)

mapping_df = pd.read_excel(
    gcms_source_file,
    sheet_name=MAPPING_SHEET,
    dtype={CAS_COLUMN: str}
)

sensory_df = pd.read_excel(
    SENSORY_CACHE_FILE,
    sheet_name=SENSORY_SHEET,
    dtype={CAS_COLUMN: str}
)

print()
print("Dimensioni matrice delle aree:", areas_df.shape)
print("Dimensioni mapping:", mapping_df.shape)
print("Dimensioni cache sensoriale:", sensory_df.shape)


# =====================================================================
# 7. CONTROLLO DELLE COLONNE OBBLIGATORIE
# =====================================================================

check_required_columns(
    areas_df,
    [
        SAMPLE_COLUMN,
        REPLICATE_COLUMN
    ],
    AREAS_SHEET
)

check_required_columns(
    mapping_df,
    [
        MAPPING_NAME_COLUMN,
        IUPAC_COLUMN,
        CAS_COLUMN,
        ROLE_COLUMN
    ],
    MAPPING_SHEET
)

check_required_columns(
    sensory_df,
    [
        CAS_COLUMN,
        THRESHOLD_COLUMN
    ],
    SENSORY_SHEET
)


# =====================================================================
# 8. PULIZIA DEL MAPPING
# =====================================================================

mapping_df[MAPPING_NAME_COLUMN] = (
    mapping_df[MAPPING_NAME_COLUMN]
    .apply(clean_text)
)

mapping_df[IUPAC_COLUMN] = (
    mapping_df[IUPAC_COLUMN]
    .apply(clean_text)
)

mapping_df[CAS_COLUMN] = (
    mapping_df[CAS_COLUMN]
    .apply(normalize_cas)
)

mapping_df["Compound_role_normalized"] = (
    mapping_df[ROLE_COLUMN]
    .apply(normalize_role)
)


# =====================================================================
# 9. IDENTIFICAZIONE DELLO STANDARD INTERNO
# =====================================================================

internal_standard_labels = {
    "internal standard",
    "internalstandard",
    "standard interno",
    "internal std",
    "is"
}

internal_standard_rows = mapping_df.loc[
    mapping_df["Compound_role_normalized"].isin(
        internal_standard_labels
    )
].copy()

if internal_standard_rows.empty:
    raise ValueError(
        "Nel foglio Compound_Mapping non è stato identificato "
        "alcuno standard interno."
    )

if len(internal_standard_rows) > 1:
    raise ValueError(
        "Nel foglio Compound_Mapping sono presenti più standard "
        "interni. Questa versione gestisce un solo standard interno."
    )

internal_standard_column = internal_standard_rows.iloc[0][
    MAPPING_NAME_COLUMN
]

if internal_standard_column not in areas_df.columns:
    raise ValueError(
        f"La colonna dello standard interno "
        f"'{internal_standard_column}' non è presente nel foglio "
        f"'{AREAS_SHEET}'."
    )

print()
print("Standard interno identificato:", internal_standard_column)


# =====================================================================
# 10. IDENTIFICAZIONE DEGLI ANALITI
# =====================================================================

analyte_labels = {
    "analyte",
    "analita",
    "compound",
    "voc"
}

analyte_mapping = mapping_df.loc[
    mapping_df["Compound_role_normalized"].isin(
        analyte_labels
    )
].copy()

if analyte_mapping.empty:
    raise ValueError(
        "Nel foglio Compound_Mapping non sono stati identificati "
        "analiti."
    )

analyte_mapping = analyte_mapping.drop_duplicates(
    subset=[MAPPING_NAME_COLUMN],
    keep="first"
)

analyte_columns = analyte_mapping[
    MAPPING_NAME_COLUMN
].tolist()

missing_area_columns = [
    column
    for column in analyte_columns
    if column not in areas_df.columns
]

if missing_area_columns:
    raise ValueError(
        "Le seguenti colonne indicate nel mapping non sono presenti "
        "nel foglio GCMS_Areas: "
        + ", ".join(missing_area_columns)
    )

print("Analiti identificati:", len(analyte_columns))

for compound in analyte_columns:
    print(" -", compound)


# =====================================================================
# 11. CONTROLLO DELLE COLONNE NON MAPPATE
# =====================================================================

metadata_columns = {
    SAMPLE_COLUMN,
    REPLICATE_COLUMN
}

mapped_area_columns = set(
    analyte_columns + [internal_standard_column]
)

unmapped_columns = [
    column
    for column in areas_df.columns
    if column not in metadata_columns
    and column not in mapped_area_columns
]

if unmapped_columns:
    print()
    print(
        "ATTENZIONE: le seguenti colonne non sono presenti nel "
        "Compound_Mapping e non verranno elaborate:"
    )

    for column in unmapped_columns:
        print(" -", column)


# =====================================================================
# 12. CONVERSIONE DELLE AREE IN VALORI NUMERICI
# =====================================================================

numeric_area_columns = (
    analyte_columns
    + [internal_standard_column]
)

for column in numeric_area_columns:

    original_non_empty = (
        areas_df[column]
        .notna()
        .sum()
    )

    areas_df[column] = pd.to_numeric(
        areas_df[column],
        errors="coerce"
    )

    converted_non_empty = (
        areas_df[column]
        .notna()
        .sum()
    )

    if converted_non_empty < original_non_empty:
        print(
            f"Attenzione: nella colonna '{column}' alcuni valori "
            "non numerici sono stati convertiti in dato mancante."
        )


# =====================================================================
# 13. PULIZIA DEGLI IDENTIFICATIVI DEI CAMPIONI
# =====================================================================

areas_df[SAMPLE_COLUMN] = (
    areas_df[SAMPLE_COLUMN]
    .apply(clean_text)
)

areas_df[REPLICATE_COLUMN] = pd.to_numeric(
    areas_df[REPLICATE_COLUMN],
    errors="coerce"
)

if areas_df[SAMPLE_COLUMN].isna().any():
    number_missing = int(
        areas_df[SAMPLE_COLUMN]
        .isna()
        .sum()
    )

    raise ValueError(
        f"Sono presenti {number_missing} righe prive di Sample_ID."
    )

if areas_df[REPLICATE_COLUMN].isna().any():
    print(
        "ATTENZIONE: alcune repliche sono mancanti o non numeriche."
    )


# =====================================================================
# 14. TRASFORMAZIONE DAL FORMATO LARGO AL FORMATO LUNGO
# =====================================================================

long_df = areas_df.melt(
    id_vars=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN,
        internal_standard_column
    ],
    value_vars=analyte_columns,
    var_name=MAPPING_NAME_COLUMN,
    value_name="Area"
)

long_df = long_df.rename(
    columns={
        internal_standard_column: "IS_area"
    }
)

print()
print(
    "Righe create nella tabella lunga:",
    len(long_df)
)


# =====================================================================
# 15. ABBINAMENTO CON IL COMPOUND MAPPING
# =====================================================================

mapping_for_merge = analyte_mapping[
    [
        MAPPING_NAME_COLUMN,
        IUPAC_COLUMN,
        CAS_COLUMN
    ]
].copy()

master_df = long_df.merge(
    mapping_for_merge,
    on=MAPPING_NAME_COLUMN,
    how="left",
    validate="many_to_one"
)

if master_df[CAS_COLUMN].isna().any():

    missing_names = (
        master_df.loc[
            master_df[CAS_COLUMN].isna(),
            MAPPING_NAME_COLUMN
        ]
        .dropna()
        .unique()
        .tolist()
    )

    raise ValueError(
        "Non è stato possibile associare il CAS alle colonne: "
        + ", ".join(missing_names)
    )


# =====================================================================
# 16. CALCOLO DELLE AREE NORMALIZZATE
# =====================================================================

valid_area = (
    master_df["Area"].notna()
    &
    (master_df["Area"] >= 0)
)

valid_internal_standard = (
    master_df["IS_area"].notna()
    &
    (master_df["IS_area"] > 0)
)

master_df["Normalized_area"] = np.where(
    valid_area & valid_internal_standard,
    master_df["Area"] / master_df["IS_area"],
    np.nan
)

master_df["Log10_normalized_area"] = np.where(
    master_df["Normalized_area"] > 0,
    np.log10(master_df["Normalized_area"]),
    np.nan
)


# =====================================================================
# 17. PULIZIA DELLA CACHE SENSORIALE
# =====================================================================

sensory_df[CAS_COLUMN] = (
    sensory_df[CAS_COLUMN]
    .apply(normalize_cas)
)

sensory_df[THRESHOLD_COLUMN] = pd.to_numeric(
    sensory_df[THRESHOLD_COLUMN],
    errors="coerce"
)

duplicated_cas = (
    sensory_df.loc[
        sensory_df[CAS_COLUMN].duplicated(keep=False),
        CAS_COLUMN
    ]
    .dropna()
    .unique()
    .tolist()
)

if duplicated_cas:
    print()
    print(
        "ATTENZIONE: nella cache sensoriale sono presenti "
        "CAS duplicati."
    )
    print(
        "Per ciascun CAS verrà utilizzata la prima riga."
    )

    for cas in duplicated_cas:
        print(" -", cas)

    sensory_df = sensory_df.drop_duplicates(
        subset=[CAS_COLUMN],
        keep="first"
    )


# =====================================================================
# 18. SELEZIONE DEI CAMPI SENSORIALI
# =====================================================================

preferred_sensory_columns = [
    CAS_COLUMN,
    "Common_name",
    "Molecular_formula",
    "Molecular_weight_g_mol",
    "Threshold_air_found",
    "Odor_threshold_air_value_original",
    "Odor_threshold_air_unit_original",
    THRESHOLD_COLUMN,
    "Threshold_conversion_status",
    "Threshold_type",
    "Threshold_conditions",
    "Odor_descriptors",
    "Flavor_descriptors",
    "Sensory_family",
    "Threshold_source_name",
    "Threshold_source_url",
    "Threshold_reference",
    "Descriptor_source_names",
    "Descriptor_source_urls",
    "Confidence",
    "Review_required",
    "Notes",
    "Retrieved_on"
]

sensory_columns_to_merge = [
    column
    for column in preferred_sensory_columns
    if column in sensory_df.columns
]

sensory_for_merge = sensory_df[
    sensory_columns_to_merge
].copy()


# =====================================================================
# 19. ABBINAMENTO TRAMITE CAS
# =====================================================================

master_df = master_df.merge(
    sensory_for_merge,
    on=CAS_COLUMN,
    how="left",
    validate="many_to_one"
)


# =====================================================================
# 20. COLONNE DI CONTROLLO QUALITÀ
# =====================================================================

sensory_record_condition = (
    master_df[THRESHOLD_COLUMN].notna()
)

if "Odor_descriptors" in master_df.columns:
    sensory_record_condition = (
        sensory_record_condition
        |
        master_df["Odor_descriptors"].notna()
    )

master_df["Sensory_record_found"] = np.where(
    sensory_record_condition,
    "Sì",
    "No"
)

master_df["Normalized_area_calculable"] = np.where(
    master_df["Normalized_area"].notna(),
    "Sì",
    "No"
)

master_df["Processing_note"] = ""

master_df.loc[
    master_df["Area"].isna(),
    "Processing_note"
] = "Area analita mancante o non numerica"

master_df.loc[
    master_df["IS_area"].isna(),
    "Processing_note"
] = "Area standard interno mancante o non numerica"

master_df.loc[
    master_df["IS_area"].notna()
    & (master_df["IS_area"] <= 0),
    "Processing_note"
] = "Area standard interno uguale o inferiore a zero"

master_df.loc[
    master_df["Area"] == 0,
    "Processing_note"
] = "Composto non rilevato: area uguale a zero"

master_df.loc[
    master_df[THRESHOLD_COLUMN].isna()
    & (master_df["Processing_note"] == ""),
    "Processing_note"
] = "Soglia olfattiva in aria non disponibile"

master_df.loc[
    master_df[THRESHOLD_COLUMN].notna()
    & (master_df[THRESHOLD_COLUMN] <= 0),
    "Processing_note"
] = "Soglia olfattiva uguale o inferiore a zero"


# =====================================================================
# 21. ORDINAMENTO
# =====================================================================

master_df = master_df.sort_values(
    by=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN,
        MAPPING_NAME_COLUMN
    ],
    ascending=[
        True,
        True,
        True
    ]
).reset_index(drop=True)


# =====================================================================
# 22. ORDINE DELLE COLONNE
# =====================================================================

main_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    MAPPING_NAME_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    "Area",
    "IS_area",
    "Normalized_area",
    "Log10_normalized_area"
]

sensory_output_columns = [
    "Common_name",
    "Molecular_formula",
    "Molecular_weight_g_mol",
    "Threshold_air_found",
    "Odor_threshold_air_value_original",
    "Odor_threshold_air_unit_original",
    THRESHOLD_COLUMN,
    "Threshold_conversion_status",
    "Threshold_type",
    "Threshold_conditions",
    "Odor_descriptors",
    "Flavor_descriptors",
    "Sensory_family",
    "Threshold_source_name",
    "Threshold_source_url",
    "Threshold_reference",
    "Descriptor_source_names",
    "Descriptor_source_urls",
    "Confidence",
    "Review_required",
    "Notes",
    "Retrieved_on"
]

quality_columns = [
    "Sensory_record_found",
    "Normalized_area_calculable",
    "Processing_note"
]

ordered_columns = (
    main_columns
    + [
        column
        for column in sensory_output_columns
        if column in master_df.columns
    ]
    + quality_columns
)

master_df = master_df[
    [
        column
        for column in ordered_columns
        if column in master_df.columns
    ]
]


# =====================================================================
# 23. TABELLA DI RIEPILOGO
# =====================================================================

summary_df = pd.DataFrame(
    {
        "Indicatore": [
            "File GC-MS utilizzato",
            "Cache sensoriale utilizzata",
            "Numero di righe/iniezioni nel file GC-MS",
            "Numero di campioni distinti",
            "Numero di analiti",
            "Numero di righe nella Master Table",
            "Aree normalizzate calcolate",
            "Record con soglia in aria disponibile",
            "Record senza soglia in aria",
            "Standard interno utilizzato"
        ],
        "Valore": [
            gcms_source_file.name,
            SENSORY_CACHE_FILE.name,
            len(areas_df),
            areas_df[SAMPLE_COLUMN].nunique(),
            len(analyte_columns),
            len(master_df),
            int(
                master_df["Normalized_area"]
                .notna()
                .sum()
            ),
            int(
                master_df[THRESHOLD_COLUMN]
                .notna()
                .sum()
            ),
            int(
                master_df[THRESHOLD_COLUMN]
                .isna()
                .sum()
            ),
            internal_standard_column
        ]
    }
)


# =====================================================================
# 24. TABELLA DI REVISIONE
# =====================================================================

review_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    MAPPING_NAME_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    "Area",
    "IS_area",
    "Normalized_area",
    THRESHOLD_COLUMN,
    "Confidence",
    "Review_required",
    "Processing_note"
]

review_columns = [
    column
    for column in review_columns
    if column in master_df.columns
]

review_mask = (
    master_df["Processing_note"] != ""
)

if "Review_required" in master_df.columns:
    review_mask = (
        review_mask
        |
        master_df["Review_required"]
        .fillna(False)
        .astype(bool)
    )

review_df = master_df.loc[
    review_mask,
    review_columns
].copy()


# =====================================================================
# 25. MATRICE DELLE AREE NORMALIZZATE
# =====================================================================

normalized_matrix = master_df.pivot_table(
    index=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN
    ],
    columns=MAPPING_NAME_COLUMN,
    values="Normalized_area",
    aggfunc="first"
)


# =====================================================================
# 26. SALVATAGGIO DEL FILE EXCEL
# =====================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="xlsxwriter"
) as writer:

    master_df.to_excel(
        writer,
        sheet_name="Master_Table",
        index=False
    )

    normalized_matrix.to_excel(
        writer,
        sheet_name="Normalized_Areas"
    )

    summary_df.to_excel(
        writer,
        sheet_name="Processing_Summary",
        index=False
    )

    review_df.to_excel(
        writer,
        sheet_name="Manual_Review",
        index=False
    )

    mapping_df.to_excel(
        writer,
        sheet_name="Mapping_Used",
        index=False
    )

    workbook = writer.book

    header_format = workbook.add_format(
        {
            "bold": True,
            "bg_color": "#D9EAF7",
            "border": 1,
            "text_wrap": True,
            "valign": "top"
        }
    )

    wrap_format = workbook.add_format(
        {
            "text_wrap": True,
            "valign": "top"
        }
    )

    integer_format = workbook.add_format(
        {
            "num_format": "0",
            "valign": "top"
        }
    )

    decimal_format = workbook.add_format(
        {
            "num_format": "0.000000",
            "valign": "top"
        }
    )

    scientific_format = workbook.add_format(
        {
            "num_format": "0.000E+00",
            "valign": "top"
        }
    )

    url_format = workbook.add_format(
        {
            "font_color": "blue",
            "underline": True,
            "text_wrap": True,
            "valign": "top"
        }
    )

    dataframe_by_sheet = {
        "Master_Table": master_df,
        "Processing_Summary": summary_df,
        "Manual_Review": review_df,
        "Mapping_Used": mapping_df
    }

    for sheet_name, dataframe in dataframe_by_sheet.items():

        worksheet = writer.sheets[sheet_name]

        worksheet.freeze_panes(1, 0)
        worksheet.set_default_row(24)

        if len(dataframe.columns) > 0:
            worksheet.autofilter(
                0,
                0,
                max(len(dataframe), 1),
                len(dataframe.columns) - 1
            )

        for column_index, column_name in enumerate(
            dataframe.columns
        ):

            worksheet.write(
                0,
                column_index,
                column_name,
                header_format
            )

            column_width = calculate_column_width(
                dataframe,
                column_name
            )

            worksheet.set_column(
                column_index,
                column_index,
                column_width,
                wrap_format
            )

    master_ws = writer.sheets["Master_Table"]

    for column_name in [
        "Area",
        "IS_area"
    ]:
        if column_name in master_df.columns:

            column_index = master_df.columns.get_loc(
                column_name
            )

            master_ws.set_column(
                column_index,
                column_index,
                14,
                integer_format
            )

    for column_name in [
        "Normalized_area",
        "Log10_normalized_area"
    ]:
        if column_name in master_df.columns:

            column_index = master_df.columns.get_loc(
                column_name
            )

            master_ws.set_column(
                column_index,
                column_index,
                18,
                decimal_format
            )

    if THRESHOLD_COLUMN in master_df.columns:

        column_index = master_df.columns.get_loc(
            THRESHOLD_COLUMN
        )

        master_ws.set_column(
            column_index,
            column_index,
            19,
            scientific_format
        )

    for column_name in [
        "Threshold_source_url",
        "Descriptor_source_urls"
    ]:
        if column_name in master_df.columns:

            column_index = master_df.columns.get_loc(
                column_name
            )

            master_ws.set_column(
                column_index,
                column_index,
                40,
                url_format
            )

    normalized_ws = writer.sheets[
        "Normalized_Areas"
    ]

    normalized_ws.freeze_panes(1, 2)


# =====================================================================
# 27. RISULTATO E DOWNLOAD
# =====================================================================

print()
print("=" * 72)
print("ELABORAZIONE COMPLETATA")
print("=" * 72)

print("File creato:", OUTPUT_FILE.name)
print("File GC-MS utilizzato:", gcms_source_file.name)
print("Cache utilizzata:", SENSORY_CACHE_FILE.name)
print("Righe della Master Table:", len(master_df))
print("Campioni distinti:", areas_df[SAMPLE_COLUMN].nunique())
print("Analiti elaborati:", len(analyte_columns))

print(
    "Aree normalizzate calcolate:",
    int(
        master_df["Normalized_area"]
        .notna()
        .sum()
    )
)

print(
    "Record con soglia in aria:",
    int(
        master_df[THRESHOLD_COLUMN]
        .notna()
        .sum()
    )
)

print(
    "Record da controllare:",
    len(review_df)
)

print()
print("Anteprima della Master Table:")

preview_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    MAPPING_NAME_COLUMN,
    CAS_COLUMN,
    "Area",
    "IS_area",
    "Normalized_area",
    "Log10_normalized_area",
    THRESHOLD_COLUMN,
    "Sensory_family",
    "Confidence",
    "Review_required",
    "Processing_note"
]

preview_columns = [
    column
    for column in preview_columns
    if column in master_df.columns
]

display(
    master_df[
        preview_columns
    ].head(20)
)

files.download(
    str(OUTPUT_FILE)
)

RICERCA DEI FILE NELLA SESSIONE COLAB
File sorgente GC-MS: GCMS_Areas.xlsx
Cache sensoriale: Sensory_Cache.xlsx

Fogli del file GC-MS: ['GCMS_Areas', 'Compound_Mapping', 'metodo_analitico']
Fogli della cache: ['Sensory_Cache', 'Threshold_Sources', 'Descriptor_Sources', 'Manual_Review', 'Input_Compounds', 'Processing_Summary']

Dimensioni matrice delle aree: (6, 7)
Dimensioni mapping: (5, 4)
Dimensioni cache sensoriale: (4, 31)

Standard interno identificato: Internal_standard
Analiti identificati: 4
 - 2-methoxyphenol
 - linalool
 - 2,3,5-trimethylpyrazine
 - phenol

Righe create nella tabella lunga: 24

ELABORAZIONE COMPLETATA
File creato: GCMS_Master_Table.xlsx
File GC-MS utilizzato: GCMS_Areas.xlsx
Cache utilizzata: Sensory_Cache.xlsx
Righe della Master Table: 24
Campioni distinti: 2
Analiti elaborati: 4
Aree normalizzate calcolate: 24
Record con soglia in aria: 24
Record da controllare: 18

Anteprima della Master Table:


,Sample_ID,Replicate,GCMS_column_name,CAS,Area,IS_area,Normalized_area,Log10_normalized_area,Odor_threshold_air_ug_m3,Sensory_family,Confidence,Review_required,Processing_note
0,Cacao_01,1,"2,3,5-trimethylpyrazine",14667-55-1,845230,510000,1.657314,0.219405,50.000000,Tostato | Caffè | Frutta secca | Cacao/Cioccol...,medium,True,
1,Cacao_01,1,2-methoxyphenol,90-05-1,125300,510000,0.245686,-0.609619,0.084000,Affumicato | Tostato | Dolce | Speziato | Legn...,high,False,
2,Cacao_01,1,linalool,78-70-6,64900,510000,0.127255,-0.895325,3.200000,Floreale | Fruttato | Dolce | Verde/Erbaceo | ...,medium,True,
3,Cacao_01,1,phenol,108-95-2,94900,510000,0.186078,-0.730304,42.339877,Fenolico/Medicinale | Chimico/Solvente | Dolce,medium,True,
4,Cacao_01,2,"2,3,5-trimethylpyrazine",14667-55-1,861500,506000,1.702569,0.231105,50.000000,Tostato | Caffè | Frutta secca | Cacao/Cioccol...,medium,True,
5,Cacao_01,2,2-methoxyphenol,90-05-1,128100,506000,0.253162,-0.596601,0.084000,Affumicato | Tostato | Dolce | Speziato | Legn...,high,False,
6,Cacao_01,2,linalool,78-70-6,63200,506000,0.124901,-0.903433,3.200000,Floreale | Fruttato | Dolce | Verde/Erbaceo | ...,medium,True,
7,Cacao_01,2,phenol,108-95-2,93200,506000,0.184190,-0.734735,42.339877,Fenolico/Medicinale | Chimico/Solvente | Dolce,medium,True,
8,Cacao_01,3,"2,3,5-trimethylpyrazine",14667-55-1,837600,514000,1.629572,0.212074,50.000000,Tostato | Caffè | Frutta secca | Cacao/Cioccol...,medium,True,
9,Cacao_01,3,2-methoxyphenol,90-05-1,121900,514000,0.237160,-0.624959,0.084000,Affumicato | Tostato | Dolce | Speziato | Legn...,high,False,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =====================================================================
# TERZA CELLA — CALCOLO DI IPA, LIPA E RANKING
#
# INPUT:
#   - variabile master_df prodotta dalla cella 2
#     oppure
#   - /content/GCMS_Master_Table.xlsx, foglio Master_Table
#
# OUTPUT:
#   /content/GCMS_IPA_Results.xlsx
#
# Nessun nuovo caricamento di file.
# Nessuna chiamata API.
# =====================================================================


# =====================================================================
# 0. INSTALLAZIONE DEL MOTORE EXCEL, SE NECESSARIO
# =====================================================================

!pip -q install xlsxwriter


# =====================================================================
# 1. IMPORTAZIONI
# =====================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import xlsxwriter

from google.colab import files


# =====================================================================
# 2. PARAMETRI
# =====================================================================

WORKING_DIRECTORY = Path("/content")

MASTER_FILE = WORKING_DIRECTORY / "GCMS_Master_Table.xlsx"
MASTER_SHEET = "Master_Table"

OUTPUT_FILE = WORKING_DIRECTORY / "GCMS_IPA_Results.xlsx"

SAMPLE_COLUMN = "Sample_ID"
REPLICATE_COLUMN = "Replicate"

COMPOUND_COLUMN = "GCMS_column_name"
IUPAC_COLUMN = "IUPAC_name"
CAS_COLUMN = "CAS"

NORMALIZED_AREA_COLUMN = "Normalized_area"
THRESHOLD_COLUMN = "Odor_threshold_air_ug_m3"

# Soglia convenzionale di riferimento.
# Deve avere la stessa unità delle soglie della cache.
REFERENCE_THRESHOLD_UG_M3 = 1.0


# =====================================================================
# 3. FUNZIONI DI SUPPORTO
# =====================================================================

def check_required_columns(dataframe, required_columns, table_name):
    """
    Verifica che siano presenti tutte le colonne obbligatorie.
    """
    missing = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing:
        raise ValueError(
            f"Nella tabella '{table_name}' mancano le colonne: "
            + ", ".join(missing)
        )


def calculate_column_width(dataframe, column_name, maximum=45):
    """
    Calcola una larghezza leggibile per una colonna Excel.
    """
    header_length = len(str(column_name))

    if dataframe.empty:
        return min(max(header_length + 2, 12), maximum)

    maximum_value_length = (
        dataframe[column_name]
        .fillna("")
        .astype(str)
        .map(len)
        .max()
    )

    width = max(
        header_length,
        int(maximum_value_length)
    ) + 2

    return min(max(width, 12), maximum)


def classify_lipa(value):
    """
    Classificazione operativa preliminare.

    Non rappresenta una classificazione sensoriale assoluta:
    serve soltanto a facilitare la lettura e il filtraggio.
    """
    if pd.isna(value):
        return "Non calcolabile"

    if value >= 3:
        return "Estremamente elevata"

    if value >= 2:
        return "Molto elevata"

    if value >= 1:
        return "Elevata"

    if value >= 0:
        return "Moderata"

    if value >= -1:
        return "Bassa"

    return "Molto bassa"


def coefficient_of_variation(mean_value, sd_value):
    """
    Calcola CV% evitando divisioni per zero.
    """
    if (
        pd.isna(mean_value)
        or pd.isna(sd_value)
        or mean_value == 0
    ):
        return np.nan

    return 100.0 * sd_value / mean_value


# =====================================================================
# 4. RECUPERO DELLA MASTER TABLE
# =====================================================================

print("=" * 72)
print("RECUPERO DELLA MASTER TABLE")
print("=" * 72)

# Se master_df esiste già nella memoria della sessione,
# ne viene utilizzata una copia.
if (
    "master_df" in globals()
    and isinstance(master_df, pd.DataFrame)
    and not master_df.empty
):

    ipa_df = master_df.copy()

    print(
        "È stata utilizzata la variabile master_df "
        "presente nella memoria della sessione."
    )

else:

    if not MASTER_FILE.exists():
        raise FileNotFoundError(
            f"Non è disponibile la variabile master_df e non è stato "
            f"trovato il file '{MASTER_FILE.name}' in /content.\n\n"
            "Eseguire prima la cella 2 nella stessa sessione Colab."
        )

    master_excel = pd.ExcelFile(
        MASTER_FILE
    )

    if MASTER_SHEET not in master_excel.sheet_names:
        raise ValueError(
            f"Nel file '{MASTER_FILE.name}' manca il foglio "
            f"'{MASTER_SHEET}'."
        )

    ipa_df = pd.read_excel(
        MASTER_FILE,
        sheet_name=MASTER_SHEET,
        dtype={CAS_COLUMN: str}
    )

    print(
        f"È stato letto il file '{MASTER_FILE.name}', "
        f"foglio '{MASTER_SHEET}'."
    )

print("Righe disponibili:", len(ipa_df))


# =====================================================================
# 5. CONTROLLO DELLE COLONNE OBBLIGATORIE
# =====================================================================

required_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    COMPOUND_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    "Area",
    "IS_area",
    NORMALIZED_AREA_COLUMN,
    THRESHOLD_COLUMN
]

check_required_columns(
    ipa_df,
    required_columns,
    MASTER_SHEET
)


# =====================================================================
# 6. CONVERSIONE DELLE COLONNE NUMERICHE
# =====================================================================

numeric_columns = [
    "Area",
    "IS_area",
    NORMALIZED_AREA_COLUMN,
    THRESHOLD_COLUMN
]

for column in numeric_columns:
    ipa_df[column] = pd.to_numeric(
        ipa_df[column],
        errors="coerce"
    )

ipa_df[REPLICATE_COLUMN] = pd.to_numeric(
    ipa_df[REPLICATE_COLUMN],
    errors="coerce"
)


# =====================================================================
# 7. CONTROLLO DELLA SOGLIA DI RIFERIMENTO
# =====================================================================

if REFERENCE_THRESHOLD_UG_M3 <= 0:
    raise ValueError(
        "REFERENCE_THRESHOLD_UG_M3 deve essere maggiore di zero."
    )


# =====================================================================
# 8. VALIDITÀ DEI DATI PER IL CALCOLO
# =====================================================================

valid_normalized_area = (
    ipa_df[NORMALIZED_AREA_COLUMN].notna()
    &
    (ipa_df[NORMALIZED_AREA_COLUMN] > 0)
)

valid_threshold = (
    ipa_df[THRESHOLD_COLUMN].notna()
    &
    (ipa_df[THRESHOLD_COLUMN] > 0)
)

ipa_df["IPA_calculable"] = np.where(
    valid_normalized_area & valid_threshold,
    "Sì",
    "No"
)


# =====================================================================
# 9. CALCOLO DI IPA
# =====================================================================

ipa_df["IPA"] = np.where(
    valid_normalized_area & valid_threshold,

    ipa_df[NORMALIZED_AREA_COLUMN]
    * REFERENCE_THRESHOLD_UG_M3
    / ipa_df[THRESHOLD_COLUMN],

    np.nan
)


# =====================================================================
# 10. CALCOLO DI LIPA
# =====================================================================

ipa_df["LIPA"] = np.where(
    ipa_df["IPA"] > 0,
    np.log10(ipa_df["IPA"]),
    np.nan
)


# =====================================================================
# 11. SCOMPOSIZIONE DEL LIPA
#
# LIPA = contributo analitico + contributo della soglia
# =====================================================================

ipa_df["Log10_analytical_response"] = np.where(
    ipa_df[NORMALIZED_AREA_COLUMN] > 0,
    np.log10(
        ipa_df[NORMALIZED_AREA_COLUMN]
    ),
    np.nan
)

ipa_df["Log10_odor_potency"] = np.where(
    ipa_df[THRESHOLD_COLUMN] > 0,

    np.log10(
        REFERENCE_THRESHOLD_UG_M3
        / ipa_df[THRESHOLD_COLUMN]
    ),

    np.nan
)


# =====================================================================
# 12. CLASSE OPERATIVA
# =====================================================================

ipa_df["IPA_priority_class"] = (
    ipa_df["LIPA"].apply(classify_lipa)
)


# =====================================================================
# 13. NOTE SUL CALCOLO
# =====================================================================

ipa_df["IPA_processing_note"] = ""

ipa_df.loc[
    ipa_df[NORMALIZED_AREA_COLUMN].isna(),
    "IPA_processing_note"
] = "Area normalizzata non disponibile"

ipa_df.loc[
    ipa_df[NORMALIZED_AREA_COLUMN].notna()
    & (ipa_df[NORMALIZED_AREA_COLUMN] <= 0),
    "IPA_processing_note"
] = "Area normalizzata uguale o inferiore a zero"

ipa_df.loc[
    ipa_df[THRESHOLD_COLUMN].isna(),
    "IPA_processing_note"
] = "Soglia olfattiva in aria non disponibile"

ipa_df.loc[
    ipa_df[THRESHOLD_COLUMN].notna()
    & (ipa_df[THRESHOLD_COLUMN] <= 0),
    "IPA_processing_note"
] = "Soglia olfattiva uguale o inferiore a zero"


# =====================================================================
# 14. RANKING ENTRO CIASCUNA REPLICA
# =====================================================================

ranking_group_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN
]

ipa_df["IPA_rank_replica"] = (
    ipa_df
    .groupby(
        ranking_group_columns,
        dropna=False
    )["LIPA"]
    .rank(
        method="dense",
        ascending=False,
        na_option="bottom"
    )
)

ipa_df["IPA_rank_replica"] = (
    ipa_df["IPA_rank_replica"]
    .astype("Int64")
)


# =====================================================================
# 15. ORDINAMENTO DELLA TABELLA A LIVELLO DI REPLICA
# =====================================================================

ipa_df = ipa_df.sort_values(
    by=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN,
        "IPA_rank_replica",
        COMPOUND_COLUMN
    ],
    ascending=[
        True,
        True,
        True,
        True
    ],
    na_position="last"
).reset_index(drop=True)


# =====================================================================
# 16. RIEPILOGO PER CAMPIONE
#
# Le aree normalizzate vengono mediate tra le repliche.
# L'IPA del campione viene calcolato sulla media delle aree normalizzate.
# =====================================================================

sample_group_columns = [
    SAMPLE_COLUMN,
    COMPOUND_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN
]

aggregation_dictionary = {
    NORMALIZED_AREA_COLUMN: [
        "count",
        "mean",
        "std",
        "min",
        "max"
    ],
    THRESHOLD_COLUMN: "first"
}

# Conserva anche i principali dati sensoriali, se presenti.
optional_first_columns = [
    "Common_name",
    "Threshold_type",
    "Odor_descriptors",
    "Flavor_descriptors",
    "Sensory_family",
    "Confidence",
    "Review_required",
    "Threshold_source_name",
    "Threshold_source_url"
]

for column in optional_first_columns:
    if column in ipa_df.columns:
        aggregation_dictionary[column] = "first"

sample_summary = (
    ipa_df
    .groupby(
        sample_group_columns,
        dropna=False
    )
    .agg(aggregation_dictionary)
    .reset_index()
)


# =====================================================================
# 17. APPIATTIMENTO DELLE INTESTAZIONI DEL RIEPILOGO
# =====================================================================

flattened_columns = []

for column in sample_summary.columns:

    if isinstance(column, tuple):

        first_part = str(column[0]).strip()
        second_part = str(column[1]).strip()

        if second_part:
            flattened_columns.append(
                f"{first_part}_{second_part}"
            )
        else:
            flattened_columns.append(first_part)

    else:
        flattened_columns.append(str(column))

sample_summary.columns = flattened_columns


# Rinomina le colonne principali.
rename_summary_columns = {
    f"{NORMALIZED_AREA_COLUMN}_count": "Number_of_replicates",
    f"{NORMALIZED_AREA_COLUMN}_mean": "Mean_normalized_area",
    f"{NORMALIZED_AREA_COLUMN}_std": "SD_normalized_area",
    f"{NORMALIZED_AREA_COLUMN}_min": "Min_normalized_area",
    f"{NORMALIZED_AREA_COLUMN}_max": "Max_normalized_area",
    f"{THRESHOLD_COLUMN}_first": THRESHOLD_COLUMN
}

for column in optional_first_columns:
    rename_summary_columns[
        f"{column}_first"
    ] = column

sample_summary = sample_summary.rename(
    columns=rename_summary_columns
)


# =====================================================================
# 18. COEFFICIENTE DI VARIAZIONE DELLE REPLICHE
# =====================================================================

sample_summary["CV_normalized_area_percent"] = (
    sample_summary.apply(
        lambda row: coefficient_of_variation(
            row["Mean_normalized_area"],
            row["SD_normalized_area"]
        ),
        axis=1
    )
)


# =====================================================================
# 19. IPA E LIPA SULLA MEDIA DELLE REPLICHE
# =====================================================================

valid_sample_mean = (
    sample_summary["Mean_normalized_area"].notna()
    &
    (sample_summary["Mean_normalized_area"] > 0)
)

valid_sample_threshold = (
    sample_summary[THRESHOLD_COLUMN].notna()
    &
    (sample_summary[THRESHOLD_COLUMN] > 0)
)

sample_summary["IPA_mean"] = np.where(
    valid_sample_mean & valid_sample_threshold,

    sample_summary["Mean_normalized_area"]
    * REFERENCE_THRESHOLD_UG_M3
    / sample_summary[THRESHOLD_COLUMN],

    np.nan
)

sample_summary["LIPA_mean"] = np.where(
    sample_summary["IPA_mean"] > 0,
    np.log10(
        sample_summary["IPA_mean"]
    ),
    np.nan
)

sample_summary["IPA_priority_class"] = (
    sample_summary["LIPA_mean"]
    .apply(classify_lipa)
)


# =====================================================================
# 20. RANKING ENTRO CIASCUN CAMPIONE
# =====================================================================

sample_summary["IPA_rank_sample"] = (
    sample_summary
    .groupby(
        SAMPLE_COLUMN,
        dropna=False
    )["LIPA_mean"]
    .rank(
        method="dense",
        ascending=False,
        na_option="bottom"
    )
)

sample_summary["IPA_rank_sample"] = (
    sample_summary["IPA_rank_sample"]
    .astype("Int64")
)


sample_summary = sample_summary.sort_values(
    by=[
        SAMPLE_COLUMN,
        "IPA_rank_sample",
        COMPOUND_COLUMN
    ],
    ascending=[
        True,
        True,
        True
    ],
    na_position="last"
).reset_index(drop=True)


# =====================================================================
# 21. MATRICE IPA PER REPLICA
# =====================================================================

ipa_matrix_replica = ipa_df.pivot_table(
    index=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN
    ],
    columns=COMPOUND_COLUMN,
    values="IPA",
    aggfunc="first"
)


# =====================================================================
# 22. MATRICE LIPA PER REPLICA
# =====================================================================

lipa_matrix_replica = ipa_df.pivot_table(
    index=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN
    ],
    columns=COMPOUND_COLUMN,
    values="LIPA",
    aggfunc="first"
)


# =====================================================================
# 23. MATRICE IPA SULLA MEDIA DEL CAMPIONE
# =====================================================================

ipa_matrix_sample = sample_summary.pivot_table(
    index=SAMPLE_COLUMN,
    columns=COMPOUND_COLUMN,
    values="IPA_mean",
    aggfunc="first"
)


# =====================================================================
# 24. MATRICE LIPA SULLA MEDIA DEL CAMPIONE
# =====================================================================

lipa_matrix_sample = sample_summary.pivot_table(
    index=SAMPLE_COLUMN,
    columns=COMPOUND_COLUMN,
    values="LIPA_mean",
    aggfunc="first"
)


# =====================================================================
# 25. TOP COMPOUNDS PER CAMPIONE
# =====================================================================

TOP_N = 20

top_compounds = sample_summary.loc[
    sample_summary["IPA_rank_sample"].notna()
    &
    (
        sample_summary["IPA_rank_sample"]
        <= TOP_N
    )
].copy()


# =====================================================================
# 26. RECORD NON CALCOLABILI
# =====================================================================

not_calculable = ipa_df.loc[
    ipa_df["IPA"].isna()
].copy()

not_calculable_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    COMPOUND_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    NORMALIZED_AREA_COLUMN,
    THRESHOLD_COLUMN,
    "IPA_processing_note"
]

not_calculable_columns = [
    column
    for column in not_calculable_columns
    if column in not_calculable.columns
]

not_calculable = not_calculable[
    not_calculable_columns
]


# =====================================================================
# 27. RIEPILOGO DELL'ELABORAZIONE
# =====================================================================

processing_summary = pd.DataFrame(
    {
        "Indicatore": [
            "Soglia di riferimento T_r (µg/m³)",
            "Numero di righe analita-replica",
            "Numero di campioni distinti",
            "Numero di analiti distinti",
            "Valori IPA calcolati",
            "Valori IPA non calcolabili",
            "Numero di righe nel riepilogo campione",
            "Top N utilizzato"
        ],
        "Valore": [
            REFERENCE_THRESHOLD_UG_M3,
            len(ipa_df),
            ipa_df[SAMPLE_COLUMN].nunique(),
            ipa_df[CAS_COLUMN].nunique(),
            int(
                ipa_df["IPA"]
                .notna()
                .sum()
            ),
            int(
                ipa_df["IPA"]
                .isna()
                .sum()
            ),
            len(sample_summary),
            TOP_N
        ]
    }
)


# =====================================================================
# 28. ORDINE DELLE COLONNE DEL RISULTATO A LIVELLO DI REPLICA
# =====================================================================

main_replica_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    "IPA_rank_replica",
    COMPOUND_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    "Area",
    "IS_area",
    NORMALIZED_AREA_COLUMN,
    THRESHOLD_COLUMN,
    "IPA",
    "LIPA",
    "Log10_analytical_response",
    "Log10_odor_potency",
    "IPA_priority_class"
]

sensory_replica_columns = [
    "Threshold_type",
    "Odor_descriptors",
    "Flavor_descriptors",
    "Sensory_family",
    "Confidence",
    "Review_required",
    "Threshold_source_name",
    "Threshold_source_url"
]

quality_replica_columns = [
    "IPA_calculable",
    "IPA_processing_note"
]

ordered_replica_columns = (
    main_replica_columns
    + [
        column
        for column in sensory_replica_columns
        if column in ipa_df.columns
    ]
    + quality_replica_columns
)

# Conserva alla fine eventuali colonne originali non incluse sopra.
remaining_columns = [
    column
    for column in ipa_df.columns
    if column not in ordered_replica_columns
]

ipa_df = ipa_df[
    ordered_replica_columns
    + remaining_columns
]


# =====================================================================
# 29. SALVATAGGIO DEL FILE EXCEL
# =====================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="xlsxwriter"
) as writer:

    ipa_df.to_excel(
        writer,
        sheet_name="IPA_by_Replicate",
        index=False
    )

    sample_summary.to_excel(
        writer,
        sheet_name="IPA_by_Sample",
        index=False
    )

    top_compounds.to_excel(
        writer,
        sheet_name="Top_Compounds",
        index=False
    )

    ipa_matrix_replica.to_excel(
        writer,
        sheet_name="IPA_Matrix_Replicate"
    )

    lipa_matrix_replica.to_excel(
        writer,
        sheet_name="LIPA_Matrix_Replicate"
    )

    ipa_matrix_sample.to_excel(
        writer,
        sheet_name="IPA_Matrix_Sample"
    )

    lipa_matrix_sample.to_excel(
        writer,
        sheet_name="LIPA_Matrix_Sample"
    )

    not_calculable.to_excel(
        writer,
        sheet_name="IPA_Not_Calculable",
        index=False
    )

    processing_summary.to_excel(
        writer,
        sheet_name="Processing_Summary",
        index=False
    )

    # -----------------------------------------------------------------
    # Formati Excel
    # -----------------------------------------------------------------

    workbook = writer.book

    header_format = workbook.add_format(
        {
            "bold": True,
            "bg_color": "#D9EAF7",
            "border": 1,
            "text_wrap": True,
            "valign": "top"
        }
    )

    wrap_format = workbook.add_format(
        {
            "text_wrap": True,
            "valign": "top"
        }
    )

    integer_format = workbook.add_format(
        {
            "num_format": "0",
            "valign": "top"
        }
    )

    decimal_format = workbook.add_format(
        {
            "num_format": "0.000000",
            "valign": "top"
        }
    )

    scientific_format = workbook.add_format(
        {
            "num_format": "0.000E+00",
            "valign": "top"
        }
    )

    rank_format = workbook.add_format(
        {
            "num_format": "0",
            "align": "center",
            "valign": "top"
        }
    )

    url_format = workbook.add_format(
        {
            "font_color": "blue",
            "underline": True,
            "text_wrap": True,
            "valign": "top"
        }
    )

    # Colori usati soltanto per la formattazione condizionale.
    high_format = workbook.add_format(
        {
            "bg_color": "#C6EFCE",
            "font_color": "#006100"
        }
    )

    medium_format = workbook.add_format(
        {
            "bg_color": "#FFEB9C",
            "font_color": "#9C6500"
        }
    )

    low_format = workbook.add_format(
        {
            "bg_color": "#FFC7CE",
            "font_color": "#9C0006"
        }
    )


    dataframe_by_sheet = {
        "IPA_by_Replicate": ipa_df,
        "IPA_by_Sample": sample_summary,
        "Top_Compounds": top_compounds,
        "IPA_Not_Calculable": not_calculable,
        "Processing_Summary": processing_summary
    }

    for sheet_name, dataframe in dataframe_by_sheet.items():

        worksheet = writer.sheets[sheet_name]

        worksheet.freeze_panes(1, 0)
        worksheet.set_default_row(24)

        if len(dataframe.columns) > 0:

            worksheet.autofilter(
                0,
                0,
                max(len(dataframe), 1),
                len(dataframe.columns) - 1
            )

        for column_index, column_name in enumerate(
            dataframe.columns
        ):

            worksheet.write(
                0,
                column_index,
                column_name,
                header_format
            )

            width = calculate_column_width(
                dataframe,
                column_name
            )

            worksheet.set_column(
                column_index,
                column_index,
                width,
                wrap_format
            )


    # -----------------------------------------------------------------
    # Formattazione IPA_by_Replicate
    # -----------------------------------------------------------------

    replica_ws = writer.sheets[
        "IPA_by_Replicate"
    ]

    for column_name in [
        "Area",
        "IS_area"
    ]:

        if column_name in ipa_df.columns:

            column_index = ipa_df.columns.get_loc(
                column_name
            )

            replica_ws.set_column(
                column_index,
                column_index,
                14,
                integer_format
            )

    for column_name in [
        NORMALIZED_AREA_COLUMN,
        THRESHOLD_COLUMN,
        "IPA"
    ]:

        if column_name in ipa_df.columns:

            column_index = ipa_df.columns.get_loc(
                column_name
            )

            replica_ws.set_column(
                column_index,
                column_index,
                18,
                scientific_format
            )

    for column_name in [
        "LIPA",
        "Log10_analytical_response",
        "Log10_odor_potency"
    ]:

        if column_name in ipa_df.columns:

            column_index = ipa_df.columns.get_loc(
                column_name
            )

            replica_ws.set_column(
                column_index,
                column_index,
                18,
                decimal_format
            )

    if "IPA_rank_replica" in ipa_df.columns:

        column_index = ipa_df.columns.get_loc(
            "IPA_rank_replica"
        )

        replica_ws.set_column(
            column_index,
            column_index,
            12,
            rank_format
        )

    if "Threshold_source_url" in ipa_df.columns:

        column_index = ipa_df.columns.get_loc(
            "Threshold_source_url"
        )

        replica_ws.set_column(
            column_index,
            column_index,
            40,
            url_format
        )

    if "LIPA" in ipa_df.columns and len(ipa_df) > 0:

        lipa_col = ipa_df.columns.get_loc("LIPA")

        replica_ws.conditional_format(
            1,
            lipa_col,
            len(ipa_df),
            lipa_col,
            {
                "type": "3_color_scale",
                "min_color": "#F8696B",
                "mid_color": "#FFEB84",
                "max_color": "#63BE7B"
            }
        )


    # -----------------------------------------------------------------
    # Formattazione IPA_by_Sample
    # -----------------------------------------------------------------

    sample_ws = writer.sheets[
        "IPA_by_Sample"
    ]

    for column_name in [
        "Mean_normalized_area",
        "SD_normalized_area",
        "Min_normalized_area",
        "Max_normalized_area",
        THRESHOLD_COLUMN,
        "IPA_mean"
    ]:

        if column_name in sample_summary.columns:

            column_index = (
                sample_summary.columns.get_loc(
                    column_name
                )
            )

            sample_ws.set_column(
                column_index,
                column_index,
                18,
                scientific_format
            )

    for column_name in [
        "LIPA_mean",
        "CV_normalized_area_percent"
    ]:

        if column_name in sample_summary.columns:

            column_index = (
                sample_summary.columns.get_loc(
                    column_name
                )
            )

            sample_ws.set_column(
                column_index,
                column_index,
                18,
                decimal_format
            )

    if "IPA_rank_sample" in sample_summary.columns:

        column_index = (
            sample_summary.columns.get_loc(
                "IPA_rank_sample"
            )
        )

        sample_ws.set_column(
            column_index,
            column_index,
            12,
            rank_format
        )

    if "LIPA_mean" in sample_summary.columns and len(sample_summary) > 0:

        lipa_mean_col = (
            sample_summary.columns.get_loc(
                "LIPA_mean"
            )
        )

        sample_ws.conditional_format(
            1,
            lipa_mean_col,
            len(sample_summary),
            lipa_mean_col,
            {
                "type": "3_color_scale",
                "min_color": "#F8696B",
                "mid_color": "#FFEB84",
                "max_color": "#63BE7B"
            }
        )


    # -----------------------------------------------------------------
    # Formattazione delle matrici
    # -----------------------------------------------------------------

    matrix_sheet_names = [
        "IPA_Matrix_Replicate",
        "LIPA_Matrix_Replicate",
        "IPA_Matrix_Sample",
        "LIPA_Matrix_Sample"
    ]

    for sheet_name in matrix_sheet_names:

        worksheet = writer.sheets[sheet_name]

        worksheet.freeze_panes(1, 2)

        worksheet.set_column(
            0,
            1,
            18
        )

        worksheet.set_column(
            2,
            200,
            16,
            scientific_format
            if sheet_name.startswith("IPA_")
            else decimal_format
        )


# =====================================================================
# 30. RISULTATO E DOWNLOAD
# =====================================================================

print()
print("=" * 72)
print("CALCOLO IPA COMPLETATO")
print("=" * 72)

print("File creato:", OUTPUT_FILE.name)

print(
    "Righe analita-replica:",
    len(ipa_df)
)

print(
    "Valori IPA calcolati:",
    int(
        ipa_df["IPA"]
        .notna()
        .sum()
    )
)

print(
    "Valori IPA non calcolabili:",
    int(
        ipa_df["IPA"]
        .isna()
        .sum()
    )
)

print(
    "Campioni distinti:",
    ipa_df[SAMPLE_COLUMN].nunique()
)

print(
    "Analiti distinti:",
    ipa_df[CAS_COLUMN].nunique()
)

print()
print("Anteprima del ranking medio per campione:")

preview_columns = [
    SAMPLE_COLUMN,
    "IPA_rank_sample",
    COMPOUND_COLUMN,
    CAS_COLUMN,
    "Mean_normalized_area",
    THRESHOLD_COLUMN,
    "IPA_mean",
    "LIPA_mean",
    "IPA_priority_class",
    "Sensory_family",
    "Confidence"
]

preview_columns = [
    column
    for column in preview_columns
    if column in sample_summary.columns
]

display(
    sample_summary[
        preview_columns
    ].head(30)
)

files.download(
    str(OUTPUT_FILE)
)

RECUPERO DELLA MASTER TABLE
È stata utilizzata la variabile master_df presente nella memoria della sessione.
Righe disponibili: 24

CALCOLO IPA COMPLETATO
File creato: GCMS_IPA_Results.xlsx
Righe analita-replica: 24
Valori IPA calcolati: 24
Valori IPA non calcolabili: 0
Campioni distinti: 2
Analiti distinti: 4

Anteprima del ranking medio per campione:


,Sample_ID,IPA_rank_sample,GCMS_column_name,CAS,Mean_normalized_area,Odor_threshold_air_ug_m3,IPA_mean,LIPA_mean,IPA_priority_class,Sensory_family,Confidence
0,Cacao_01,1,2-methoxyphenol,90-05-1,0.245336,0.084000,2.920666,0.465482,Moderata,Affumicato | Tostato | Dolce | Speziato | Legn...,high
1,Cacao_01,2,linalool,78-70-6,0.126724,3.200000,0.039601,-1.402292,Molto bassa,Floreale | Fruttato | Dolce | Verde/Erbaceo | ...,medium
2,Cacao_01,3,"2,3,5-trimethylpyrazine",14667-55-1,1.663152,50.000000,0.033263,-1.478038,Molto bassa,Tostato | Caffè | Frutta secca | Cacao/Cioccol...,medium
3,Cacao_01,4,phenol,108-95-2,0.185550,42.339877,0.004382,-2.358289,Molto bassa,Fenolico/Medicinale | Chimico/Solvente | Dolce,medium
4,Cacao_02,1,2-methoxyphenol,90-05-1,0.199480,0.084000,2.374760,0.375620,Moderata,Affumicato | Tostato | Dolce | Speziato | Legn...,high
5,Cacao_02,2,linalool,78-70-6,0.145738,3.200000,0.045543,-1.341576,Molto bassa,Floreale | Fruttato | Dolce | Verde/Erbaceo | ...,medium
6,Cacao_02,3,"2,3,5-trimethylpyrazine",14667-55-1,1.845060,50.000000,0.036901,-1.432959,Molto bassa,Tostato | Caffè | Frutta secca | Cacao/Cioccol...,medium
7,Cacao_02,4,phenol,108-95-2,0.253445,42.339877,0.005986,-2.222867,Molto bassa,Fenolico/Medicinale | Chimico/Solvente | Dolce,medium


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>